# Temporal Motif Feature Extraction for Ethereum Fraud Detection

## 1. Import Libraries and Load Data

In [1]:
import polars as pl
from pathlib import Path
from collections import defaultdict
from bisect import bisect_left, bisect_right
import pandas as pd

In [2]:
# Load community definitions, labeled fraud data, and transaction logs
infomap = pd.read_csv(Path("infomap_communities.csv"))  # Cluster allocations computed by the Infomap algorithm
fraud = pd.read_csv(Path("all_detected_fraud_accounts.csv"))  # Verified malicious account registry (ground-truth dataset)
df = pl.read_csv("eth_tx_last4days_clean.csv")  # Main collection of parsed Ethereum transaction ledgers

## 2. Data Preprocessing

Cast the columns to their proper formats, standardize string hashes, extract native numeric unix timestamps, convert raw transaction sizes from Wei to Ether, and strictly sort chronologically to enable windowed streaming operations.

In [3]:
events = (
    df
    .select([
        pl.col("hash"),
        pl.col("from_address").str.to_lowercase().alias("source"),
        pl.col("to_address").str.to_lowercase().alias("target"),
        pl.col("value").alias("value_wei"),
        pl.col("block_timestamp")
    ])
    .with_columns([
        pl.col("target").fill_null("__contract_creation__"),
        (
            pl.col("block_timestamp")
            .str.to_datetime(format="%Y-%m-%d %H:%M:%S%z")
            # Construct standard temporal Unix timestamps (seconds)
            .dt.timestamp("ms") // 1000
        ).alias("timestamp"),
        (
            pl.col("value_wei").cast(pl.Float64, strict=False) / 1_000_000_000_000_000_000
        # Convert the numeric value to standard Ether scales
        ).alias("value_eth")
    ])
    .select([
        "hash",
        "source",
        "target",
        "timestamp",
        "block_timestamp",
        "value_wei",
        "value_eth"
    ])
    # Enforce absolute sequential ordering by transaction time
    .sort("timestamp")
)


In [4]:
# Verify shape, schema, and look for any remaining missing values
print(events.shape)
print(events.schema)

# Run extensive structural data assertions and print descriptive aggregations
events.select([
    pl.col("timestamp").null_count().alias("missing_timestamp"),
    pl.col("source").null_count().alias("missing_source"),
    pl.col("target").null_count().alias("missing_target"),
    pl.col("value_wei").null_count().alias("missing_value_wei"),
    pl.col("value_eth").null_count().alias("missing_value_eth"),
    pl.col("timestamp").min().alias("min_timestamp"),
    pl.col("timestamp").max().alias("max_timestamp"),
    pl.col("block_timestamp").first().alias("first_block_timestamp"),
    pl.col("block_timestamp").last().alias("last_block_timestamp"),
    pl.col("value_eth").min().alias("min_value_eth"),
    pl.col("value_eth").max().alias("max_value_eth"),
    pl.col("value_eth").mean().alias("mean_value_eth")
])

(4290480, 7)
Schema([('hash', String), ('source', String), ('target', String), ('timestamp', Int64), ('block_timestamp', String), ('value_wei', Float64), ('value_eth', Float64)])


missing_timestamp,missing_source,missing_target,missing_value_wei,missing_value_eth,min_timestamp,max_timestamp,first_block_timestamp,last_block_timestamp,min_value_eth,max_value_eth,mean_value_eth
u32,u32,u32,u32,u32,i64,i64,str,str,f64,f64,f64
0,0,0,0,0,1762546907,1762793675,"""2025-11-07 20:21:47+00:00""","""2025-11-10 16:54:35+00:00""",0.0,39448.75,0.688368


## 3. Community Selection

Aggregate duplicated illicit accounts to construct flat records with unique node addresses, concatenate multi-type behavior labels, and map them explicitly onto Infomap network indices to discover high-density anomaly targets.

In [5]:
# Deduplicate fraud data and combine multi-type labels per address
fraud_clean = (
    fraud
    .groupby("address", as_index=False)
    .agg({
        # Unify label properties safely
        "detected_type": lambda x: ";".join(sorted(set(x.dropna().astype(str)))),
        # Select the default first mapping node
        "node_id": "first"
    })
)

# Explicit class definition assignment for targets
fraud_clean["label"] = "fraud"

print("Fraud clean shape:", fraud_clean.shape)
display(fraud_clean.head())

Fraud clean shape: (4351, 4)


,address,detected_type,node_id,label
0,0x0000000000000000000000000000000000000000,phishing,1,fraud
1,0x0000000000000000000000000000000000001002,phishing,4,fraud
2,0x00000000000000447e69651d841bd8d104bed493,phishing,8,fraud
3,0x000000000000f10286d9c1c5d4635a25070572b6,phishing,37,fraud
4,0x0000000000771a79d0fc7f3b7fe270eb4498f20b,phishing,71,fraud


In [6]:
# Left merge structural node partitions with clean fraud classifications
nodes = infomap.merge(
    fraud_clean[["address", "label", "detected_type"]],
    on="address",
    how="left"
)

# Populate missing missing labels explicitly as standard 'normal' profiles
nodes["label"] = nodes["label"].fillna("normal")
nodes["detected_type"] = nodes["detected_type"].fillna("normal")

print("Nodes shape:", nodes.shape)

print(nodes["label"].value_counts())

Nodes shape: (1686534, 4)
label
normal    1682526
fraud        4008
Name: count, dtype: int64


In [7]:
# Compute core metrics (total sizes, actual fraud node occurrences, and density indices) for each community cluster
community_stats = (
    nodes
    .groupby("infomap_id")
    .agg(
        n_nodes=("address", "count"),
        n_fraud_nodes=("label", lambda x: (x == "fraud").sum()),
        n_normal_nodes=("label", lambda x: (x == "normal").sum())
    )
    .reset_index()
)

# Calculate ratio percentages and sort by density scale metrics
community_stats["fraud_rate"] = (
    community_stats["n_fraud_nodes"] / community_stats["n_nodes"]
)

community_stats = community_stats.sort_values(
    ["n_fraud_nodes", "n_nodes"],
    ascending=[False, False]
)

display(community_stats.head(20))

,infomap_id,n_nodes,n_fraud_nodes,n_normal_nodes,fraud_rate
845,846,2820,459,2361,0.162766
20,21,265670,57,265613,0.000215
389,390,17500,46,17454,0.002629
241,242,5901,28,5873,0.004745
1139,1140,187,26,161,0.139037
2900,2901,175,26,149,0.148571
277,278,7853,16,7837,0.002037
494,495,5104,16,5088,0.003135
1035,1036,774,14,760,0.018088
246,247,5825,13,5812,0.002232


In [8]:
# Isolate verified illicit entities and track distinct crime profiles inside clusters
fraud_nodes = nodes[nodes["label"] == "fraud"].copy()

fraud_type_by_community = (
    fraud_nodes
    .groupby(["infomap_id", "detected_type"])
    .agg(
        n_fraud_type=("address", "count")
    )
    .reset_index()
)

# Sort distributions progressively based on criminal category types
fraud_type_by_community = fraud_type_by_community.sort_values(
    ["detected_type", "n_fraud_type"],
    ascending=[True, False]
)


In [9]:
for fraud_type in fraud_nodes["detected_type"].unique():
    print("=" * 80)
    print("Fraud type:", fraud_type)
    
    temp = (
        fraud_nodes[fraud_nodes["detected_type"] == fraud_type]
        .groupby("infomap_id")
        .agg(
            n_this_type=("address", "count")
        )
        .reset_index()
        .merge(
            community_stats[["infomap_id", "n_nodes", "n_fraud_nodes", "n_normal_nodes", "fraud_rate"]],
            on="infomap_id",
            how="left"
        )
        .sort_values("n_this_type", ascending=False)
    )
    
    display(temp.head(10))

Fraud type: phishing


,infomap_id,n_this_type,n_nodes,n_fraud_nodes,n_normal_nodes,fraud_rate
91,390,45,17500,46,17454,0.002629
4,21,22,265670,57,265613,0.000215
119,495,16,5104,16,5088,0.003135
202,1036,13,774,14,760,0.018088
117,484,12,281,12,269,0.042705
57,247,11,5825,13,5812,0.002232
262,1620,9,39,9,30,0.230769
51,222,8,84785,11,84774,0.000130
21,127,8,542,8,534,0.014760
56,242,7,5901,28,5873,0.004745


Fraud type: mixer


,infomap_id,n_this_type,n_nodes,n_fraud_nodes,n_normal_nodes,fraud_rate
3,21,27,265670,57,265613,0.000215
23,242,17,5901,28,5873,0.004745
31,278,14,7853,16,7837,0.002037
530,22339,10,36,12,24,0.333333
99,782,9,3377,9,3368,0.002665
42,314,8,6028,8,6020,0.001327
13,165,6,12,7,5,0.583333
125,1135,5,2485,10,2475,0.004024
77,573,5,2660,6,2654,0.002256
50,372,5,6545,8,6537,0.001222


Fraud type: wash_trading


...

,infomap_id,n_this_type,n_nodes,n_fraud_nodes,n_normal_nodes,fraud_rate
1,279,3,4540,7,4533,0.001542
0,247,2,5825,13,5812,0.002232
28,17010,1,44,1,43,0.022727
27,16223,1,35,1,34,0.028571
26,15655,1,39,2,37,0.051282
25,10448,1,37,1,36,0.027027
24,9638,1,56,1,55,0.017857
23,9333,1,125,3,122,0.024000
22,9053,1,36,1,35,0.027778
21,8207,1,20,1,19,0.050000


Fraud type: mixer;phishing


,infomap_id,n_this_type,n_nodes,n_fraud_nodes,n_normal_nodes,fraud_rate
0,263,1,12047,8,12039,0.000664
1,294,1,1719,2,1717,0.001163
18,20917,1,10,1,9,0.100000
17,14777,1,13,2,11,0.153846
16,14135,1,9,2,7,0.222222
15,12184,1,99,1,98,0.010101
14,10312,1,21,3,18,0.142857
13,8399,1,163,1,162,0.006135
12,6996,1,133,1,132,0.007519
11,3576,1,243,2,241,0.008230


Fraud type: mixer;phishing;wash_trading


,infomap_id,n_this_type,n_nodes,n_fraud_nodes,n_normal_nodes,fraud_rate
0,291,1,2183,2,2181,0.000916
1,971,1,184,2,182,0.010870
2,5573,1,90,1,89,0.011111
3,8531,1,46,2,44,0.043478


In [10]:
selected_communities = {
    "wash_trading": 846,
    "phishing": 390,
    "mixer": 242
}

selected_node_lists = {}

for name, community_id in selected_communities.items():
    temp = nodes[nodes["infomap_id"] == community_id].copy()
    selected_node_lists[name] = temp
    
    print("=" * 80)
    print(name)
    print("infomap_id:", community_id)
    print("n_nodes:", len(temp))
    print("\nLabel counts:")
    print(temp["label"].value_counts())
    print("\nDetected type counts:")
    print(temp[temp["label"] == "fraud"]["detected_type"].value_counts())
    
    output_name = f"selected_{name}_community_{community_id}_nodes.csv"
    temp.to_csv(output_name, index=False)
    print("\nSaved:", output_name)

wash_trading
infomap_id: 846
n_nodes: 2820

Label counts:
label
normal    2361
fraud      459
Name: count, dtype: int64

Detected type counts:
detected_type
wash_trading          455
phishing                2
mixer;wash_trading      1
mixer                   1
Name: count, dtype: int64

Saved: selected_wash_trading_community_846_nodes.csv
phishing
infomap_id: 390
n_nodes: 17500

Label counts:
label
normal    17454
fraud        46
Name: count, dtype: int64

Detected type counts:
detected_type
phishing    45
mixer        1
Name: count, dtype: int64

Saved: selected_phishing_community_390_nodes.csv
mixer
infomap_id: 242
n_nodes: 5901

Label counts:
label
normal    5873
fraud       28
Name: count, dtype: int64

Detected type counts:
detected_type
mixer                 17
phishing               7
wash_trading           3
mixer;wash_trading     1
Name: count, dtype: int64

Saved: selected_mixer_community_242_nodes.csv



## 4. Temporal Window Selection

Conduct exploratory data analysis using temporal sliding windows across sample interaction pairs to discover empirical delays, evaluate motif frequency variations, and choose the optimal Δ interval.

In [11]:
motif_sample = (
    events
    .filter(pl.col("target") != "__contract_creation__")
    .filter(pl.col("source") != pl.col("target"))
    .select([
        "hash",
        "source",
        "target",
        "timestamp",
        "block_timestamp",
        "value_wei"
    ])
    .sort("timestamp")
)

motif_sample.shape

(4235659, 6)

In [12]:
motif_sample.select([
    pl.col("timestamp").min().alias("min_timestamp"),
    pl.col("timestamp").max().alias("max_timestamp"),
    ((pl.col("timestamp").max() - pl.col("timestamp").min()) / 60).alias("duration_minutes"),
    pl.len().alias("n_events")
])

min_timestamp,max_timestamp,duration_minutes,n_events
i64,i64,f64,u32
1762546907,1762793675,4112.8,4235659


In [13]:
tx_per_minute = (
    motif_sample
    .with_columns((pl.col("timestamp") // 60).alias("minute_bin"))
    .group_by("minute_bin")
    .agg(pl.len().alias("n_tx"))
    .sort("minute_bin")
)

tx_per_minute.select([
    pl.col("n_tx").min().alias("min_tx_per_minute"),
    pl.col("n_tx").median().alias("median_tx_per_minute"),
    pl.col("n_tx").max().alias("max_tx_per_minute")
])

min_tx_per_minute,median_tx_per_minute,max_tx_per_minute
u32,f64,u32
168,1024.0,1984


### 4.1 Reciprocity Window Sensitivity Analysis

Evaluate the occurrence rate, total address involvement, and financial weight distribution of rapid bidirectional reciprocity motifs (A→B followed by B→A) across varying sliding window sizes (Δ).

In [14]:
# Establish a broad exploratory evaluation horizon (24 hours in seconds)
max_delta = 24 * 60 * 60

# Extract forward-directed sender properties and sort sequentially
left = (
    motif_sample
    .select([
        pl.col("hash").alias("hash_1"),
        pl.col("source").alias("a"),
        pl.col("target").alias("b"),
        pl.col("timestamp").alias("t1"),
        pl.col("value_wei").alias("value_1")
    ])
    .sort(["a", "b", "t1"])
)

# Extract inverse-directed structural lines to locate back-and-forth interactions
right = (
    motif_sample
    .select([
        pl.col("hash").alias("hash_2"),
        pl.col("target").alias("a"),
        pl.col("source").alias("b"),
        pl.col("timestamp").alias("t2"),
        pl.col("value_wei").alias("value_2")
    ])
    .sort(["a", "b", "t2"])
)

# Identify potential reciprocity structures within the maximum observation window
reciprocity_candidates = (
    left
    .join_asof(
        right,
        left_on="t1",
        right_on="t2",
        by=["a", "b"],
        strategy="forward",
        tolerance=max_delta
    )
    .filter(pl.col("t2").is_not_null())
    .filter(pl.col("t2") > pl.col("t1"))
    .with_columns((pl.col("t2") - pl.col("t1")).alias("delay_seconds"))
)

reciprocity_candidates.head()

/var/folders/3k/bctm07x95zjcttz_6djk0y180000gn/T/ipykernel_20074/2922947204.py:32: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  left


hash_1,a,b,t1,value_1,hash_2,t2,value_2,delay_seconds
str,str,str,i64,f64,str,i64,f64,i64
"""0x5b72838d2aefa5e48c9f4daf393b…","""0x000019c77100b0cb544fe48a7bde…","""0xe5962479d187a214bd81df5f43b0…",1762773719,2.9979e16,"""0xa2a14c6ce288ce62dbc01705cb62…",1762774319,3.0000e16,600
"""0xf966f1ed42a447a2d348a6236370…","""0x0003b5aa5e30e97fcc596bb5d0f3…","""0xa9ac43f5b5e38155a288d1a01d2c…",1762566659,8.2220e19,"""0x3a212819752b704da8908b21f95f…",1762571219,4.5188e20,4560
"""0x846be6329ad8239d58dd1cf70709…","""0x0003b5aa5e30e97fcc596bb5d0f3…","""0xa9ac43f5b5e38155a288d1a01d2c…",1762611083,7.1120e19,"""0xd5c90838fbbec5e90a5ce83ae451…",1762613267,4.8867e20,2184
"""0x23678c8b7ebe50705d04811278f3…","""0x0003b5aa5e30e97fcc596bb5d0f3…","""0xa9ac43f5b5e38155a288d1a01d2c…",1762613639,5.0435e19,"""0x17124575afeedf9f525dd0d49699…",1762652987,6.1506e20,39348
"""0x4d9e776a1cf0894d121f1341e2be…","""0x0003b5aa5e30e97fcc596bb5d0f3…","""0xa9ac43f5b5e38155a288d1a01d2c…",1762657043,4.9780e21,"""0xba6db302ed58bc70bc419fc4c2ad…",1762681775,2.8944e21,24732


In [15]:
# Analyze the statistical distribution of temporal delays for motif candidates
reciprocity_candidates.select([
    pl.len().alias("n_reciprocity_candidates"),
    pl.col("delay_seconds").min().alias("min_delay"),
    pl.col("delay_seconds").median().alias("median_delay"),
    pl.col("delay_seconds").quantile(0.75).alias("q75_delay"),
    pl.col("delay_seconds").quantile(0.90).alias("q90_delay"),
    pl.col("delay_seconds").quantile(0.95).alias("q95_delay"),
    pl.col("delay_seconds").quantile(0.99).alias("q99_delay"),
    pl.col("delay_seconds").max().alias("max_delay")
])

n_reciprocity_candidates,min_delay,median_delay,q75_delay,q90_delay,q95_delay,q99_delay,max_delay
u32,i64,f64,f64,f64,f64,f64,i64
69886,12,1932.0,27492.0,68808.0,83256.0,85932.0,86400


In [16]:
# Define exploratory time window buckets to evaluate distribution patterns
delta_windows = {
    "10m": 10 * 60,
    "20m": 20 * 60,
    "1h": 60 * 60,
    "6h": 6 * 60 * 60,
    "24h": 24 * 60 * 60
}

reciprocity_summary = []

for name, delta in delta_windows.items():
    temp = reciprocity_candidates.filter(pl.col("delay_seconds") <= delta)
    
    # Extract the quantitative row count representing valid reciprocity motifs found
    n_motifs = temp.height
    
    # Calculate unique participating addresses inside the current temporal subset
    n_unique_addresses = (
        pl.concat([
            temp.select(pl.col("a").alias("address")),
            temp.select(pl.col("b").alias("address"))
        ])
        .select(pl.col("address").n_unique())
        .item()
        if n_motifs > 0 else 0
    )
    
    # Aggregate the cumulative financial transaction volume of the matched pairs in Wei units
    total_weight = temp.select((pl.col("value_1") + pl.col("value_2")).sum()).item() if n_motifs > 0 else 0
    
    reciprocity_summary.append({
        "delta": name,
        "delta_seconds": delta,
        "reciprocity_motifs": n_motifs,
        "unique_addresses": n_unique_addresses,
        "total_weight_wei": total_weight
    })

reciprocity_summary = pl.DataFrame(reciprocity_summary)

reciprocity_summary

delta,delta_seconds,reciprocity_motifs,unique_addresses,total_weight_wei
str,i64,i64,i64,f64
"""10m""",600,25813,18385,2.5770e22
"""20m""",1200,30865,20418,3.1496e22
"""1h""",3600,38500,24301,4.5501e22
"""6h""",21600,50833,28722,1.7523e23
"""24h""",86400,69886,36737,3.0727e23


### 4.2 Linear Chain Motif Discovery

Prepare shifted transaction logs to isolate cascading relational transfers where node B receives funds from node A and subsequently transfers assets to a third distinct node C within the maximum lookahead window.

In [17]:
# Format the incoming left-hand side of the linear transmission chain
left_chain = (
    motif_sample
    .select([
        pl.col("hash").alias("hash_1"),
        pl.col("source").alias("a"),
        pl.col("target").alias("b"),
        pl.col("timestamp").alias("t1"),
        pl.col("value_wei").alias("value_1")
    ])
    .sort(["b", "t1"])
)

# Format the corresponding right-hand side of the linear transmission chain
right_chain = (
    motif_sample
    .select([
        pl.col("hash").alias("hash_2"),
        pl.col("source").alias("b"),
        pl.col("target").alias("c"),
        pl.col("timestamp").alias("t2"),
        pl.col("value_wei").alias("value_2")
    ])
    .sort(["b", "t2"])
)

# Identify valid linear chain sequences using an asof execution strategy
chain_candidates = (
    left_chain
    .join_asof(
        right_chain,
        left_on="t1",
        right_on="t2",
        by="b",
        strategy="forward",
        tolerance=max_delta
    )
    .filter(pl.col("t2").is_not_null())
    .filter(pl.col("t2") > pl.col("t1"))
    .filter(pl.col("a") != pl.col("c"))
    .with_columns((pl.col("t2") - pl.col("t1")).alias("delay_seconds"))
)

chain_candidates.head()

/var/folders/3k/bctm07x95zjcttz_6djk0y180000gn/T/ipykernel_20074/3432039656.py:29: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  left_chain


hash_1,a,b,t1,value_1,hash_2,c,t2,value_2,delay_seconds
str,str,str,i64,f64,str,str,i64,f64,i64
"""0x665d4cd0d13e981fdff2466eea31…","""0x8c1ddca6916451658c667a2cc8e8…","""0x00000000000007736e2f9aa5630b…",1762552451,5.2100e15,"""0xbbc290551203800b1b865fbe5fd8…","""0x1cc5c8ae16b8c5857c41981cafe9…",1762567499,1.8700e16,15048
"""0x54e2edd82d685a5aaa8b945d5906…","""0x8c1ddca6916451658c667a2cc8e8…","""0x00000000000007736e2f9aa5630b…",1762552559,2.5525e15,"""0xbbc290551203800b1b865fbe5fd8…","""0x1cc5c8ae16b8c5857c41981cafe9…",1762567499,1.8700e16,14940
"""0x886abe0bc09ed7fae01a4fef79b9…","""0x89065b675f7a11b253eb325ef0ec…","""0x00000000000007736e2f9aa5630b…",1762568183,7.4154e15,"""0xb7582ed9b90b68ca1b8cab4225ab…","""0x89b4afd880e18e31bc249d978d7f…",1762597451,5.0000e15,29268
"""0x6bd521ece31968ebc4cbbd8e5f73…","""0x7b805277add586ecca724436a445…","""0x000000000008ff37b8d9c2f8d166…",1762708595,2.0000e18,"""0x7dd8ce5efc32c7cf0d889c5babc5…","""0x4cd00e387622c35bddb9b4c962c1…",1762708691,2.0000e18,96
"""0xf9b3140bdc712cb3b540d61f1388…","""0x00643c240e93bf5b5ed7d35944dd…","""0x000000000020a61d9ad9371a410c…",1762719131,0.0,"""0x7172a10102eac060f124df225f97…","""0x2786d482f46031b8402c3ce6a29a…",1762719143,9.0460e11,12


Extract the statistical distribution characteristics of the elapsed time delay between the incoming and outgoing steps of linear chains to map out behavioral velocities.

In [18]:
chain_candidates.select([
    pl.len().alias("n_chain_candidates"),
    pl.col("delay_seconds").min().alias("min_delay"),
    pl.col("delay_seconds").median().alias("median_delay"),
    pl.col("delay_seconds").quantile(0.75).alias("q75_delay"),
    pl.col("delay_seconds").quantile(0.90).alias("q90_delay"),
    pl.col("delay_seconds").quantile(0.95).alias("q95_delay"),
    pl.col("delay_seconds").quantile(0.99).alias("q99_delay"),
    pl.col("delay_seconds").max().alias("max_delay")
])

n_chain_candidates,min_delay,median_delay,q75_delay,q90_delay,q95_delay,q99_delay,max_delay
u32,i64,f64,f64,f64,f64,f64,i64
934700,12,264.0,2004.0,24192.0,45144.0,78036.0,86400


In [19]:
# Initialize a collection array to save structural results across tested time frames
chain_summary = []

# Loop across each designated exploratory time threshold bucket
for name, delta in delta_windows.items():
    temp = chain_candidates.filter(pl.col("delay_seconds") <= delta)
    
    n_motifs = temp.height
    
    # Extract unique participating nodes across all three positions in the chain layout
    n_unique_addresses = (
        pl.concat([
            temp.select(pl.col("a").alias("address")),
            temp.select(pl.col("b").alias("address")),
            temp.select(pl.col("c").alias("address"))
        ])
        .select(pl.col("address").n_unique())
        .item()
        if n_motifs > 0 else 0
    )
    
    # Compute the total value of assets transferred across both legs of the chain combined
    total_weight = temp.select((pl.col("value_1") + pl.col("value_2")).sum()).item() if n_motifs > 0 else 0
    
    chain_summary.append({
        "delta": name,
        "delta_seconds": delta,
        "chain_motifs": n_motifs,
        "unique_addresses": n_unique_addresses,
        "total_weight_wei": total_weight
    })

chain_summary = pl.DataFrame(chain_summary)

chain_summary

delta,delta_seconds,chain_motifs,unique_addresses,total_weight_wei
str,i64,i64,i64,f64
"""10m""",600,573681,415945,3.3189e24
"""20m""",1200,660679,462022,4.0568e24
"""1h""",3600,741291,508914,4.6965e24
"""6h""",21600,836138,555915,5.1304e24
"""24h""",86400,934700,607368,5.5518e24


### 4.3 Fan-In and Fan-Out Structural Sensitivity Analysis

In [20]:
# Initialize a data repository array to store structural metrics for fan structures
fan_summary = []

# Loop through each predefined exploratory time delta parameter block
for name, delta in delta_windows.items():
    # Compute discrete time bins by performing integer division on continuous transaction timestamps
    temp = (
        motif_sample
        .with_columns((pl.col("timestamp") // delta).alias("time_bin"))
    )
    
    # Identify Fan-In structures where multiple sources transfer funds to a common target in the same time bin
    fan_in = (
        temp
        .group_by(["target", "time_bin"])
        .agg([
            pl.len().alias("n_tx"),
            pl.col("source").n_unique().alias("n_unique_sources"),
            pl.col("value_wei").sum().alias("total_value_wei")
        ])
        .filter(pl.col("n_unique_sources") >= 3)
    )
    
    # Identify Fan-Out structures where a single source splits funds among multiple targets in the same time bin
    fan_out = (
        temp
        .group_by(["source", "time_bin"])
        .agg([
            pl.len().alias("n_tx"),
            pl.col("target").n_unique().alias("n_unique_targets"),
            pl.col("value_wei").sum().alias("total_value_wei")
        ])
        .filter(pl.col("n_unique_targets") >= 3)
    )
    
    # Collect the calculated geometric shape occurrences into the comparative array
    fan_summary.append({
        "delta": name,
        "delta_seconds": delta,
        "fan_in_windows": fan_in.height,
        "fan_in_addresses": fan_in.select(pl.col("target").n_unique()).item() if fan_in.height > 0 else 0,
        "fan_out_windows": fan_out.height,
        "fan_out_addresses": fan_out.select(pl.col("source").n_unique()).item() if fan_out.height > 0 else 0
    })

fan_summary = pl.DataFrame(fan_summary)

fan_summary

delta,delta_seconds,fan_in_windows,fan_in_addresses,fan_out_windows,fan_out_addresses
str,i64,i64,i64,i64,i64
"""10m""",600,105333,7213,105790,39195
"""20m""",1200,90775,9395,106134,48433
"""1h""",3600,68831,13178,105966,60053
"""6h""",21600,45748,19728,102570,74920
"""24h""",86400,37695,25574,102137,86954


### 4.4 Repeated Same-Direction Motif Discovery

Isolate repetitive sequential transaction behaviors directed from the exact same sender address to the exact same receiver address using localized window shifting over entity pairs.

In [21]:
max_delta = 24 * 60 * 60

# Identify continuous, consecutive repetitive operations across stable directed pairs
repeat_candidates = (
    motif_sample
    .select([
        pl.col("hash").alias("hash_1"),
        pl.col("source").alias("a"),
        pl.col("target").alias("b"),
        pl.col("timestamp").alias("t1"),
        pl.col("value_wei").alias("value_1")
    ])
    .sort(["a", "b", "t1", "hash_1"])
    .with_columns([
        pl.col("hash_1").shift(-1).over(["a", "b"]).alias("hash_2"),
        pl.col("t1").shift(-1).over(["a", "b"]).alias("t2"),
        pl.col("value_1").shift(-1).over(["a", "b"]).alias("value_2")
    ])
    .filter(pl.col("t2").is_not_null())
    .filter(pl.col("t2") > pl.col("t1"))
    .with_columns(
        (pl.col("t2") - pl.col("t1")).alias("delay_seconds")
    )
    .filter(pl.col("delay_seconds") <= max_delta)
)
# Preview the initial rows of mapped same-direction repetitive transfers
repeat_candidates.head()

hash_1,a,b,t1,value_1,hash_2,t2,value_2,delay_seconds
str,str,str,i64,f64,str,i64,f64,i64
"""0xe307c411712628e22cc08db4a59f…","""0x00000000000007736e2f9aa5630b…","""0xeaf7e304ea5f0670b223d5024633…",1762620311,3.6070e15,"""0x9f407422581d664c7572db1c2cc9…",1762620599,4.0000e15,288
"""0xe6c4f7816125f111e7f690426339…","""0x0000000000000792b90728399935…","""0x0000000000efa780a8e6f50fc5de…",1762547291,6.2521345e7,"""0x8b7d1c730ac9e2d42b1fe0e6616a…",1762550255,9.4437377e7,2964
"""0x8b7d1c730ac9e2d42b1fe0e6616a…","""0x0000000000000792b90728399935…","""0x0000000000efa780a8e6f50fc5de…",1762550255,9.4437377e7,"""0x7e02bc1bf5c05ce1309240945b00…",1762551659,9.1422721e7,1404
"""0x7e02bc1bf5c05ce1309240945b00…","""0x0000000000000792b90728399935…","""0x0000000000efa780a8e6f50fc5de…",1762551659,9.1422721e7,"""0x19a37f95efbb7f9be62c1b5ffc27…",1762551863,6.0751873e7,204
"""0x19a37f95efbb7f9be62c1b5ffc27…","""0x0000000000000792b90728399935…","""0x0000000000efa780a8e6f50fc5de…",1762551863,6.0751873e7,"""0x7d0e26b2e11db9adbaf9045ea9a3…",1762552763,1.04464385e8,900


In [22]:
# Initialize a metric tracking array to map out direct repetition frequency patterns
repeat_summary = []
# Iterate across each step threshold window item inside the dictionary collection
for name, delta in delta_windows.items():
    temp = repeat_candidates.filter(pl.col("delay_seconds") <= delta)
    
    n_motifs = temp.height
    # Calculate unique participating addresses contributing to these repetitive patterns
    n_unique_addresses = (
        pl.concat([
            temp.select(pl.col("a").alias("address")),
            temp.select(pl.col("b").alias("address"))
        ])
        .select(pl.col("address").n_unique())
        .item()
        if n_motifs > 0 else 0
    )
    # Compute the total value of assets transferred across consecutive repeated operations
    total_weight = (
        temp.select((pl.col("value_1") + pl.col("value_2")).sum()).item()
        if n_motifs > 0 else 0
    )
    # Log the metrics into our collection dictionary tracking structure
    repeat_summary.append({
        "delta": name,
        "delta_seconds": delta,
        "repeat_motifs": n_motifs,
        "unique_addresses": n_unique_addresses,
        "total_weight_wei": total_weight
    })

repeat_summary = pl.DataFrame(repeat_summary)

repeat_summary

delta,delta_seconds,repeat_motifs,unique_addresses,total_weight_wei
str,i64,i64,i64,f64
"""10m""",600,994858,128279,4.7135e23
"""20m""",1200,1158715,148572,6.6622e23
"""1h""",3600,1387575,178614,1.2207e24
"""6h""",21600,1688894,267275,1.8668e24
"""24h""",86400,1870059,306467,2.5623e24


### 4.5 Directed 3-Cycle Motif Optimization

In [23]:
max_delta = 24 * 60 * 60

# Format the first leg of the 3-cycle sequence
edge_1 = (
    motif_sample
    .select([
        pl.col("hash").alias("hash_1"),
        pl.col("source").alias("a"),
        pl.col("target").alias("b"),
        pl.col("timestamp").alias("t1"),
        pl.col("value_wei").alias("value_1")
    ])
    .sort(["b", "t1"])
)

# Format the second leg of the 3-cycle sequence
edge_2 = (
    motif_sample
    .select([
        pl.col("hash").alias("hash_2"),
        pl.col("source").alias("b"),
        pl.col("target").alias("c"),
        pl.col("timestamp").alias("t2"),
        pl.col("value_wei").alias("value_2")
    ])
    .sort(["b", "t2"])
)

# Identify valid two-step paths using an asof join strategy
two_step_paths = (
    edge_1
    .join_asof(
        edge_2,
        left_on="t1",
        right_on="t2",
        by="b",
        strategy="forward",
        tolerance=max_delta
    )
    .filter(pl.col("t2").is_not_null())
    .filter(pl.col("t2") > pl.col("t1"))
    .filter(pl.col("a") != pl.col("b"))
    .filter(pl.col("b") != pl.col("c"))
    .filter(pl.col("a") != pl.col("c"))
    .with_columns((pl.col("t2") - pl.col("t1")).alias("delay_12"))
)

two_step_paths.head()

/var/folders/3k/bctm07x95zjcttz_6djk0y180000gn/T/ipykernel_20074/1203483548.py:31: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  edge_1


hash_1,a,b,t1,value_1,hash_2,c,t2,value_2,delay_12
str,str,str,i64,f64,str,str,i64,f64,i64
"""0x665d4cd0d13e981fdff2466eea31…","""0x8c1ddca6916451658c667a2cc8e8…","""0x00000000000007736e2f9aa5630b…",1762552451,5.2100e15,"""0xbbc290551203800b1b865fbe5fd8…","""0x1cc5c8ae16b8c5857c41981cafe9…",1762567499,1.8700e16,15048
"""0x54e2edd82d685a5aaa8b945d5906…","""0x8c1ddca6916451658c667a2cc8e8…","""0x00000000000007736e2f9aa5630b…",1762552559,2.5525e15,"""0xbbc290551203800b1b865fbe5fd8…","""0x1cc5c8ae16b8c5857c41981cafe9…",1762567499,1.8700e16,14940
"""0x886abe0bc09ed7fae01a4fef79b9…","""0x89065b675f7a11b253eb325ef0ec…","""0x00000000000007736e2f9aa5630b…",1762568183,7.4154e15,"""0xb7582ed9b90b68ca1b8cab4225ab…","""0x89b4afd880e18e31bc249d978d7f…",1762597451,5.0000e15,29268
"""0x6bd521ece31968ebc4cbbd8e5f73…","""0x7b805277add586ecca724436a445…","""0x000000000008ff37b8d9c2f8d166…",1762708595,2.0000e18,"""0x7dd8ce5efc32c7cf0d889c5babc5…","""0x4cd00e387622c35bddb9b4c962c1…",1762708691,2.0000e18,96
"""0xf9b3140bdc712cb3b540d61f1388…","""0x00643c240e93bf5b5ed7d35944dd…","""0x000000000020a61d9ad9371a410c…",1762719131,0.0,"""0x7172a10102eac060f124df225f97…","""0x2786d482f46031b8402c3ce6a29a…",1762719143,9.0460e11,12


In [24]:
# Format the third, closing leg of the 3-cycle sequence
edge_3 = (
    motif_sample
    .select([
        pl.col("hash").alias("hash_3"),
        pl.col("source").alias("c"),
        pl.col("target").alias("a"),
        pl.col("timestamp").alias("t3"),
        pl.col("value_wei").alias("value_3")
    ])
    .sort(["c", "a", "t3"])
)

# Complete the directed 3-cycle mapping by joining the closing leg
cycle_candidates = (
    two_step_paths
    .sort(["c", "a", "t2"])
    .join_asof(
        edge_3,
        left_on="t2",
        right_on="t3",
        by=["c", "a"],
        strategy="forward",
        tolerance=max_delta
    )
    .filter(pl.col("t3").is_not_null())
    .filter(pl.col("t3") > pl.col("t2"))
    .filter(pl.col("hash_1") != pl.col("hash_2"))
    .filter(pl.col("hash_1") != pl.col("hash_3"))
    .filter(pl.col("hash_2") != pl.col("hash_3"))
    .with_columns((pl.col("t3") - pl.col("t1")).alias("delay_seconds"))
)

cycle_candidates.head()

/var/folders/3k/bctm07x95zjcttz_6djk0y180000gn/T/ipykernel_20074/2433222471.py:16: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  two_step_paths


hash_1,a,b,t1,value_1,hash_2,c,t2,value_2,delay_12,hash_3,t3,value_3,delay_seconds
str,str,str,i64,f64,str,str,i64,f64,i64,str,i64,f64,i64
"""0x7041009b387f3412b668a20f4bb2…","""0xf584f8728b874a6a5c7a8d4d387c…","""0xbcd986563ba93af5c6c92b16ab06…",1762731767,2.3800e18,"""0x6d7172ddc36398a4796f10551607…","""0x0003b5aa5e30e97fcc596bb5d0f3…",1762732007,2.3809e18,240,"""0x3f84b288e8798a1aa1162d04cf28…",1762769303,7.3699e18,37536
"""0xeb314ad7199e6178fe00d13e5a05…","""0xf584f8728b874a6a5c7a8d4d387c…","""0xbcd986563ba93af5c6c92b16ab06…",1762734179,3.8300e19,"""0x5aa0633f7f690f210a2ff32df663…","""0x0003b5aa5e30e97fcc596bb5d0f3…",1762734407,3.8300e19,228,"""0x3f84b288e8798a1aa1162d04cf28…",1762769303,7.3699e18,35124
"""0x23367edad1863917bbf7d06808d8…","""0xef317e433b0836f294866d43f67d…","""0x2a107c3285f67a5ce030943eff36…",1762678715,1.0174e16,"""0xceea781feba398c3e17e13da3bec…","""0x004f342dec20b464a3528abe43b7…",1762678751,1.0152e16,36,"""0xc9d936741cbeee463d55c7b6b867…",1762678991,2.5427e16,276
"""0xdc334c4ca90b7864e2c11ac53bb9…","""0x07e6beb786b58cbdd8319213700b…","""0xe5feb16182f367d5d98148f363c9…",1762610987,3.9000e16,"""0xd9154ec8c96ac39cc5e6cace4729…","""0x0067cc2416f792cf0ec5629f5506…",1762632251,3.8746e16,21264,"""0x1c3fa7b3faca39898db5a1395bc4…",1762708643,1.9860e17,97656
"""0xcac208ad35f26de858115a72c96d…","""0x07e6beb786b58cbdd8319213700b…","""0xe5feb16182f367d5d98148f363c9…",1762656875,6.5000e16,"""0x42e7f0869705d8d0a48468877d6f…","""0x0067cc2416f792cf0ec5629f5506…",1762657187,6.4956e16,312,"""0x1c3fa7b3faca39898db5a1395bc4…",1762708643,1.9860e17,51768


In [25]:
cycle_summary = []

# Iterate across each step threshold window item inside the dictionary collection
for name, delta in delta_windows.items():
    temp = cycle_candidates.filter(pl.col("delay_seconds") <= delta)
    
    n_motifs = temp.height
    
    # Calculate unique participating addresses across all positions in the cycle
    n_unique_addresses = (
        pl.concat([
            temp.select(pl.col("a").alias("address")),
            temp.select(pl.col("b").alias("address")),
            temp.select(pl.col("c").alias("address"))
        ])
        .select(pl.col("address").n_unique())
        .item()
        if n_motifs > 0 else 0
    )
    
    # Compute the total value of assets transferred across all three legs of the cycle
    total_weight = (
        temp.select((pl.col("value_1") + pl.col("value_2") + pl.col("value_3")).sum()).item()
        if n_motifs > 0 else 0
    )
    
    # Record the summary metrics into the tracking dictionary array
    cycle_summary.append({
        "delta": name,
        "delta_seconds": delta,
        "cycle_motifs": n_motifs,
        "unique_addresses": n_unique_addresses,
        "total_weight_wei": total_weight
    })

cycle_summary = pl.DataFrame(cycle_summary)

cycle_summary

delta,delta_seconds,cycle_motifs,unique_addresses,total_weight_wei
str,i64,i64,i64,f64
"""10m""",600,1799,2189,1.6030e21
"""20m""",1200,2858,3226,3.5111e21
"""1h""",3600,5107,4853,1.1841e22
"""6h""",21600,8460,6743,6.6907e22
"""24h""",86400,11889,8804,1.3845e23


### 4.6 Final Parameter Configuration Selection

In [26]:
main_delta_name = "1h"
main_delta = 60 * 60

chosen_delta = {
    "main_delta_name": "1h",
    "main_delta_seconds": 60 * 60,
    "reason": "Best empirical trade-off between motif coverage and temporal specificity"
}

chosen_delta

{'main_delta_name': '1h',
 'main_delta_seconds': 3600,
 'reason': 'Best empirical trade-off between motif coverage and temporal specificity'}

Evaluating multiple structural layout formats (Reciprocity, Chains, Fans, Repetitions, and Cycles) shows that structural pattern density trends stabilize near the 1-hour (Δ=3600s) threshold. This window serves as the optimal hyperparameter configuration, providing strong signal separation for fraud detection models by capturing coordinated transactional behavior while filtering out background noise from unrelated market activity.

In [27]:
window_selection_summary = (
    reciprocity_summary
    .select(["delta", "delta_seconds", "reciprocity_motifs"])
    .join(
        chain_summary.select(["delta", "chain_motifs"]),
        on="delta",
        how="left"
    )
    .join(
        fan_summary.select(["delta", "fan_in_windows", "fan_out_windows"]),
        on="delta",
        how="left"
    )
    .join(
        repeat_summary.select(["delta", "repeat_motifs"]),
        on="delta",
        how="left"
    )
    .join(
        cycle_summary.select(["delta", "cycle_motifs"]),
        on="delta",
        how="left"
    )
)

window_selection_summary

delta,delta_seconds,reciprocity_motifs,chain_motifs,fan_in_windows,fan_out_windows,repeat_motifs,cycle_motifs
str,i64,i64,i64,i64,i64,i64,i64
"""10m""",600,25813,573681,105333,105790,994858,1799
"""20m""",1200,30865,660679,90775,106134,1158715,2858
"""1h""",3600,38500,741291,68831,105966,1387575,5107
"""6h""",21600,50833,836138,45748,102570,1688894,8460
"""24h""",86400,69886,934700,37695,102137,1870059,11889


In [28]:
window_selection_for_slide = (
    window_selection_summary
    .with_columns([
        (
            pl.col("reciprocity_motifs")
            + pl.col("chain_motifs")
            + pl.col("fan_in_windows")
            + pl.col("fan_out_windows")
            + pl.col("repeat_motifs")
            + pl.col("cycle_motifs")
        ).alias("total_motifs")
    ])
)

window_selection_for_slide

delta,delta_seconds,reciprocity_motifs,chain_motifs,fan_in_windows,fan_out_windows,repeat_motifs,cycle_motifs,total_motifs
str,i64,i64,i64,i64,i64,i64,i64,i64
"""10m""",600,25813,573681,105333,105790,994858,1799,1807274
"""20m""",1200,30865,660679,90775,106134,1158715,2858,2050026
"""1h""",3600,38500,741291,68831,105966,1387575,5107,2347270
"""6h""",21600,50833,836138,45748,102570,1688894,8460,2732643
"""24h""",86400,69886,934700,37695,102137,1870059,11889,3026366


## 5. Transaction Subgraph Preparation

In [29]:
# Define dictionary mapping the target analytical behavioral profiles to their respective node storage files
community_files = {
    "wash_trading_846": "selected_wash_trading_community_846_nodes.csv",
    "phishing_390": "selected_phishing_community_390_nodes.csv",
    "mixer_242": "selected_mixer_community_242_nodes.csv"
}

# Initialize data container dictionaries to hold parsed DataFrames and optimized lookup index sets
community_nodes = {}
community_address_sets = {}

# Iterate over each target behavioral category to load and audit structural layouts
for name, path in community_files.items():
    nodes_df = pl.read_csv(path).with_columns([
        pl.col("address").str.to_lowercase()
    ])
    
    # Cache the processed node registry DataFrame into the storage dictionary container
    community_nodes[name] = nodes_df
    
    # Extract the normalized address series and cast it into a standard flat list configuration
    addresses = (
        nodes_df
        .select("address")
        .to_series()
        .to_list()
    )
    
    # Convert list records into an optimized hash set to achieve O(1) query lookup operations during execution loops
    community_address_sets[name] = set(addresses)
    
    print("=" * 80)
    print(name)
    print("n_nodes:", len(addresses))
    
    print(
        nodes_df
        .group_by("label")
        .agg(pl.len().alias("count"))
    )

wash_trading_846
n_nodes: 2820
shape: (2, 2)
┌────────┬───────┐
│ label  ┆ count │
│ ---    ┆ ---   │
│ str    ┆ u32   │
╞════════╪═══════╡
│ fraud  ┆ 459   │
│ normal ┆ 2361  │
└────────┴───────┘
phishing_390
n_nodes: 17500
shape: (2, 2)
┌────────┬───────┐
│ label  ┆ count │
│ ---    ┆ ---   │
│ str    ┆ u32   │
╞════════╪═══════╡
│ fraud  ┆ 46    │
│ normal ┆ 17454 │
└────────┴───────┘
mixer_242
n_nodes: 5901
shape: (2, 2)
┌────────┬───────┐
│ label  ┆ count │
│ ---    ┆ ---   │
│ str    ┆ u32   │
╞════════╪═══════╡
│ fraud  ┆ 28    │
│ normal ┆ 5873  │
└────────┴───────┘


Filter the global transaction ledger to isolate activities where at least one interacting vertex intersects the target community profile set. Categorize edges dynamically into internal, outgoing, or incoming boundary type scopes based on node set constraints.

In [30]:
# Initialize data container dictionary to store isolated edge transactions for each community partition
boundary_transactions = {}

# Process each community tracking partition to parse structural subgraphs and topological boundaries
for name, address_set in community_address_sets.items():
    # Define boolean expressions checking if an interaction originates or terminates inside our target community set
    source_inside_expr = pl.col("source").is_in(address_set)
    target_inside_expr = pl.col("target").is_in(address_set)
    
    # Filter the transaction repository and append localized directional tracking columns
    tx = (
        events
        .filter(source_inside_expr | target_inside_expr)
        .with_columns([
            source_inside_expr.alias("source_inside_community"),
            target_inside_expr.alias("target_inside_community")
        ])
        .with_columns([
            # Apply conditional evaluation matching rules to group boundary connection flows
            pl.when(
                pl.col("source_inside_community") & pl.col("target_inside_community")
            )
            .then(pl.lit("internal"))
            .when(
                pl.col("source_inside_community") & ~pl.col("target_inside_community")
            )
            .then(pl.lit("outgoing"))
            .when(
                ~pl.col("source_inside_community") & pl.col("target_inside_community")
            )
            .then(pl.lit("incoming"))
            .otherwise(pl.lit("other"))
            .alias("boundary_type")
        ])
        .sort("timestamp")
    )
    
    # Save the parsed boundary transaction records dataframe to the central caching dictionary
    boundary_transactions[name] = tx
    
    # Display quantitative execution summary tables showing the distribution metrics of boundary transaction categories
    print("=" * 80)
    print(name)
    print("Boundary transactions:", tx.height)
    print(tx.group_by("boundary_type").agg(pl.len().alias("n_tx")).sort("n_tx", descending=True))
    
    display(tx.head())

wash_trading_846
Boundary transactions: 33681
shape: (3, 2)
┌───────────────┬───────┐
│ boundary_type ┆ n_tx  │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ internal      ┆ 28218 │
│ outgoing      ┆ 3927  │
│ incoming      ┆ 1536  │
└───────────────┴───────┘


hash,source,target,timestamp,block_timestamp,value_wei,value_eth,source_inside_community,target_inside_community,boundary_type
str,str,str,i64,str,f64,f64,bool,bool,str
"""0x195d31a0f3b211455edfe70b566f…","""0x1e42d03ae8b26cd81bf7997f739c…","""0x7a250d5630b4cf539739df2c5dac…",1762548023,"""2025-11-07 20:40:23+00:00""",0.0,0.0,true,false,"""outgoing"""
"""0x2f3f83ea9392924b22b5ff5a6aa1…","""0x5cece24e0085bba4ab59c24fc20d…","""0xfafe9e4b59fa4493fa8816dbabe6…",1762549331,"""2025-11-07 21:02:11+00:00""",0.0,0.0,false,true,"""incoming"""
"""0xb0383ed495733ab6a2b74d6e9b16…","""0x9af9ee552e4249a3133a8409729e…","""0x7a250d5630b4cf539739df2c5dac…",1762549487,"""2025-11-07 21:04:47+00:00""",1.4990e17,0.1499,true,false,"""outgoing"""
"""0x7174747f5d3a3e2597d546a1ebff…","""0x7ee6f0cbf1c63ea3dd791f7c815c…","""0xfafe9e4b59fa4493fa8816dbabe6…",1762550147,"""2025-11-07 21:15:47+00:00""",0.0,0.0,true,true,"""internal"""
"""0x7618f9a984ae39c86e0735d3e49a…","""0x7ee6f0cbf1c63ea3dd791f7c815c…","""0x7a250d5630b4cf539739df2c5dac…",1762550147,"""2025-11-07 21:15:47+00:00""",0.0,0.0,true,false,"""outgoing"""


phishing_390
Boundary transactions: 65504
shape: (3, 2)
┌───────────────┬───────┐
│ boundary_type ┆ n_tx  │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ internal      ┆ 28376 │
│ outgoing      ┆ 24742 │
│ incoming      ┆ 12386 │
└───────────────┴───────┘


hash,source,target,timestamp,block_timestamp,value_wei,value_eth,source_inside_community,target_inside_community,boundary_type
str,str,str,i64,str,f64,f64,bool,bool,str
"""0xf86667101d0df10bb09461c5aafb…","""0x8f8d1206d1bce12ff892731f8a14…","""0x5253e395b06ee6275dd87c69f46f…",1762546919,"""2025-11-07 20:21:59+00:00""",2.6938e16,0.026938,false,true,"""incoming"""
"""0xc6f84245f231c6a332b05032bce0…","""0x0ada3111b866ff1ad0477f0c5d2e…","""0xff05652c20ee3599ea06816a7932…",1762546919,"""2025-11-07 20:21:59+00:00""",1.3469e16,0.013469,false,true,"""incoming"""
"""0x02ab304c94e0076a55de684bcf17…","""0x8f8d1206d1bce12ff892731f8a14…","""0xd867c63027aaaff665ebfd4224c9…",1762546943,"""2025-11-07 20:22:23+00:00""",1.6831e16,0.016831,false,true,"""incoming"""
"""0x5aa6202da760798f8261dfb4d781…","""0x974caa59e49682cda0ad2bbe8298…","""0xf334a845ff38b567e764a1750456…",1762546955,"""2025-11-07 20:22:35+00:00""",5.8490e17,0.584896,true,false,"""outgoing"""
"""0x0d2f2bfffccfdfa7fe4004566678…","""0x852dc1c875f0117ea08086100108…","""0x2defec3740f6ab344f0afded500a…",1762546955,"""2025-11-07 20:22:35+00:00""",1.3087e16,0.013087,false,true,"""incoming"""


mixer_242
Boundary transactions: 70569
shape: (3, 2)
┌───────────────┬───────┐
│ boundary_type ┆ n_tx  │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ outgoing      ┆ 47765 │
│ internal      ┆ 16110 │
│ incoming      ┆ 6694  │
└───────────────┴───────┘


hash,source,target,timestamp,block_timestamp,value_wei,value_eth,source_inside_community,target_inside_community,boundary_type
str,str,str,i64,str,f64,f64,bool,bool,str
"""0x632c7aa0293d1867cd25e39a62a9…","""0x28c6c06298d514db089934071355…","""0xee7ae85f2fe2239e27d9c1e23fff…",1762546919,"""2025-11-07 20:21:59+00:00""",0.0,0.0,true,true,"""internal"""
"""0x14ac44f9320a3fcbde8cf3760c9d…","""0x28c6c06298d514db089934071355…","""0x7a58c0be72be218b41c608b7fe7c…",1762546931,"""2025-11-07 20:22:11+00:00""",0.0,0.0,true,false,"""outgoing"""
"""0xcf9a4fa88d0fe14724fcf3091f41…","""0x28c6c06298d514db089934071355…","""0xdac17f958d2ee523a22062069945…",1762546931,"""2025-11-07 20:22:11+00:00""",0.0,0.0,true,false,"""outgoing"""
"""0xa42f645f0834d6d881514b0a4903…","""0x28c6c06298d514db089934071355…","""0xd0ec028a3d21533fdd200838f39c…",1762546931,"""2025-11-07 20:22:11+00:00""",0.0,0.0,true,false,"""outgoing"""
"""0x301cd040f0636697d69296bbfd78…","""0x4cbe437aa7a83457221d657207d7…","""0x4cbe437aa7a83457221d657207d7…",1762546931,"""2025-11-07 20:22:11+00:00""",0.0,0.0,true,true,"""internal"""


In [31]:
# Export the parsed, structurally typed boundary transaction tables into external storage files
for name, tx in boundary_transactions.items():
    output_path = f"{name}_boundary_transactions.csv"
    tx.write_csv(output_path)
    print("Saved:", output_path, "rows:", tx.height)

Saved: wash_trading_846_boundary_transactions.csv rows: 33681
Saved: phishing_390_boundary_transactions.csv rows: 65504
Saved: mixer_242_boundary_transactions.csv rows: 70569


Calculate high-fidelity network summary metrics evaluating cross-boundary intersections, user nodes behavior footprints, unique entity limits, chronological ranges, and asset transfer weights.

In [32]:
# Initialize data container array to aggregate diagnostics dictionaries for baseline comparison matrix construction
boundary_diagnostics_rows = []

# Process subgraphs metrics comprehensively to build structural profiles
for name, tx in boundary_transactions.items():
    nodes_df = community_nodes[name]
    
    # Isolate known ground-truth fraudulent address hashes inside optimized unique target hash lists
    fraud_nodes = set(
        nodes_df
        .filter(pl.col("label") == "fraud")
        .select("address")
        .to_series()
        .to_list()
    )
    
    # Isolate baseline non-fraud normal address hashes inside unique target hash lists
    normal_nodes = set(
        nodes_df
        .filter(pl.col("label") == "normal")
        .select("address")
        .to_series()
        .to_list()
    )
    
    # Map out the comprehensive population reference boundaries for the targeted cluster block
    community_addresses = set(
        nodes_df
        .select("address")
        .to_series()
        .to_list()
    )
    
    # Check if any boundary interactions were successfully discovered in the processing timeline
    if tx.height > 0:
        # Extract unique participating identities found in either sending or receiving roles inside this subgraph
        tx_sources = set(tx.select("source").to_series().to_list())
        tx_targets = set(tx.select("target").to_series().to_list())
        tx_nodes = tx_sources.union(tx_targets)
        
        # Compute absolute transaction totals across each localized flow classification category
        internal_count = tx.filter(pl.col("boundary_type") == "internal").height
        incoming_count = tx.filter(pl.col("boundary_type") == "incoming").height
        outgoing_count = tx.filter(pl.col("boundary_type") == "outgoing").height
        
        # Apply set operations to differentiate between internal community vertices and external entities
        community_nodes_in_tx = tx_nodes.intersection(community_addresses)
        external_nodes_in_tx = tx_nodes.difference(community_addresses)
        
        # Populate comprehensive diagnostic descriptors parsing exact topological details
        row = {
            "community_name": name,
            "n_nodes_in_community": nodes_df.height,
            "n_fraud_nodes_in_community": len(fraud_nodes),
            "n_normal_nodes_in_community": len(normal_nodes),
            "n_boundary_transactions": tx.height,
            "n_internal_transactions": internal_count,
            "n_incoming_transactions": incoming_count,
            "n_outgoing_transactions": outgoing_count,
            "n_unique_sources": len(tx_sources),
            "n_unique_targets": len(tx_targets),
            "n_all_nodes_in_boundary_tx": len(tx_nodes),
            "n_community_nodes_in_boundary_tx": len(community_nodes_in_tx),
            "n_external_nodes_in_boundary_tx": len(external_nodes_in_tx),
            "n_fraud_nodes_in_boundary_tx": len(community_nodes_in_tx.intersection(fraud_nodes)),
            "n_normal_community_nodes_in_boundary_tx": len(community_nodes_in_tx.intersection(normal_nodes)),
            "min_timestamp": tx.select(pl.col("timestamp").min()).item(),
            "max_timestamp": tx.select(pl.col("timestamp").max()).item(),
            "total_value_eth": tx.select(pl.col("value_eth").sum()).item(),
            "mean_value_eth": tx.select(pl.col("value_eth").mean()).item()
        }
    else:
        # Assign standardized empty fallback records to handle zero activity exceptions gracefully
        row = {
            "community_name": name,
            "n_nodes_in_community": nodes_df.height,
            "n_fraud_nodes_in_community": len(fraud_nodes),
            "n_normal_nodes_in_community": len(normal_nodes),
            "n_boundary_transactions": 0,
            "n_internal_transactions": 0,
            "n_incoming_transactions": 0,
            "n_outgoing_transactions": 0,
            "n_unique_sources": 0,
            "n_unique_targets": 0,
            "n_all_nodes_in_boundary_tx": 0,
            "n_community_nodes_in_boundary_tx": 0,
            "n_external_nodes_in_boundary_tx": 0,
            "n_fraud_nodes_in_boundary_tx": 0,
            "n_normal_community_nodes_in_boundary_tx": 0,
            "min_timestamp": None,
            "max_timestamp": None,
            "total_value_eth": 0,
            "mean_value_eth": None
        }
    
    # Append the calculated metrics mapping dictionary row into the main tracker list array
    boundary_diagnostics_rows.append(row)

# Transform the accumulated row metrics dictionaries list into an operational diagnostic summary DataFrame
boundary_diagnostics_df = pl.DataFrame(boundary_diagnostics_rows)
boundary_diagnostics_df

community_name,n_nodes_in_community,n_fraud_nodes_in_community,n_normal_nodes_in_community,n_boundary_transactions,n_internal_transactions,n_incoming_transactions,n_outgoing_transactions,n_unique_sources,n_unique_targets,n_all_nodes_in_boundary_tx,n_community_nodes_in_boundary_tx,n_external_nodes_in_boundary_tx,n_fraud_nodes_in_boundary_tx,n_normal_community_nodes_in_boundary_tx,min_timestamp,max_timestamp,total_value_eth,mean_value_eth
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64
"""wash_trading_846""",2820,459,2361,33681,28218,1536,3927,3194,1922,3254,2809,445,459,2350,1762548023,1762793663,20824.774555,0.618294
"""phishing_390""",17500,46,17454,65504,28376,12386,24742,17341,14385,20499,15382,5117,46,15336,1762546919,1762793663,5169.063001,0.078912
"""mixer_242""",5901,28,5873,70569,16110,6694,47765,5657,12497,15444,4333,11111,27,4306,1762546919,1762793675,693496.526301,9.827212


## 6. Base Transaction Feature Extraction

In [33]:
def compute_boundary_node_features(boundary_transactions, community_nodes):
    all_features = []

    # Iterate over each target network cluster block to perform localized graph metrics aggregation
    for name, tx in boundary_transactions.items():
        # Retrieve the baseline community member node metadata tracking registry frame
        nodes_df = community_nodes[name]

        # Isolate edges that are completely self-contained within the localized community cluster bounds
        internal_tx = tx.filter(
            (pl.col("source_inside_community") == True) &
            (pl.col("target_inside_community") == True)
        )

        # Isolate incoming external supply lines running from outside nodes into internal cluster targets
        incoming_external_tx = tx.filter(
            (pl.col("source_inside_community") == False) &
            (pl.col("target_inside_community") == True)
        )

        # Isolate outgoing external drain lines running from internal cluster sources to outside targets
        outgoing_external_tx = tx.filter(
            (pl.col("source_inside_community") == True) &
            (pl.col("target_inside_community") == False)
        )

        # Aggregate internal outbound metrics grouping explicitly by the internal sender address node role
        internal_out = (
            internal_tx
            .group_by("source")
            .agg([
                pl.len().alias("internal_out_tx"),
                pl.col("value_eth").sum().alias("internal_out_value_eth"),
                pl.col("target").n_unique().alias("unique_internal_receivers")
            ])
            .rename({"source": "address"})
        )

        # Aggregate internal inbound metrics grouping explicitly by the internal receiver address node role
        internal_in = (
            internal_tx
            .group_by("target")
            .agg([
                pl.len().alias("internal_in_tx"),
                pl.col("value_eth").sum().alias("internal_in_value_eth"),
                pl.col("source").n_unique().alias("unique_internal_senders")
            ])
            .rename({"target": "address"})
        )

        # Aggregate external inlet flows grouping by the internal recipient target address node role
        incoming_external = (
            incoming_external_tx
            .group_by("target")
            .agg([
                pl.len().alias("incoming_from_external_tx"),
                pl.col("value_eth").sum().alias("incoming_from_external_value_eth"),
                pl.col("source").n_unique().alias("unique_external_senders")
            ])
            .rename({"target": "address"})
        )

        # Aggregate external outlet drains grouping by the internal originator source address node role
        outgoing_external = (
            outgoing_external_tx
            .group_by("source")
            .agg([
                pl.len().alias("outgoing_to_external_tx"),
                pl.col("value_eth").sum().alias("outgoing_to_external_value_eth"),
                pl.col("target").n_unique().alias("unique_external_receivers")
            ])
            .rename({"source": "address"})
        )

        # Compile and enrich the full node topology registry with the engineered local features matrix
        features = (
            nodes_df
            .join(internal_out, on="address", how="left")
            .join(internal_in, on="address", how="left")
            .join(incoming_external, on="address", how="left")
            .join(outgoing_external, on="address", how="left")
            .fill_null(0)
            .with_columns([
                pl.lit(name).alias("community"),

# Compute cumulative localized interaction frequencies on the boundary layout
                (
                    pl.col("internal_out_tx") +
                    pl.col("internal_in_tx") +
                    pl.col("incoming_from_external_tx") +
                    pl.col("outgoing_to_external_tx")
                ).alias("total_tx_boundary"),
# Compute cumulative localized financial asset volume turnovers in Ether
                (
                    pl.col("internal_out_value_eth") +
                    pl.col("internal_in_value_eth") +
                    pl.col("incoming_from_external_value_eth") +
                    pl.col("outgoing_to_external_value_eth")
                ).alias("total_value_eth_boundary"),
# Aggregate unique directional outbound destination endpoints counters
                (
                    pl.col("unique_internal_receivers") +
                    pl.col("unique_external_receivers")
                ).alias("unique_receivers_total"),
# Aggregate unique directional inbound source origin endpoints counters
                (
                    pl.col("unique_internal_senders") +
                    pl.col("unique_external_senders")
                ).alias("unique_senders_total"),

# Compute overall structural interaction degree across internal and external components
                (
                    pl.col("unique_internal_receivers") +
                    pl.col("unique_external_receivers") +
                    pl.col("unique_internal_senders") +
                    pl.col("unique_external_senders")
                ).alias("unique_counterparties_sum")
            ])
            .select([
                "community",
                "address",
                "label",
                "internal_out_tx",
                "internal_out_value_eth",
                "unique_internal_receivers",
                "internal_in_tx",
                "internal_in_value_eth",
                "unique_internal_senders",
                "incoming_from_external_tx",
                "incoming_from_external_value_eth",
                "unique_external_senders",
                "outgoing_to_external_tx",
                "outgoing_to_external_value_eth",
                "unique_external_receivers",
                "total_tx_boundary",
                "total_value_eth_boundary",
                "unique_receivers_total",
                "unique_senders_total",
                "unique_counterparties_sum"
            ])
        )

        # Store the current community's finalized structural features array into global collections list
        all_features.append(features)

        # Output operational verification summaries displaying the dimensions of the generated table
        print("=" * 80)
        print(name)
        print("features shape:", features.shape)
        display(features.head(10))
# Concat all community features matrices vertically into a single unified analytical dataframe
    nodes_communities = (
        pl.concat(all_features)
        .sort(["community", "total_tx_boundary"], descending=[False, True])
    )

    return nodes_communities

In [34]:
# Run the pipeline function to build the complete localized graph feature table
nodes_communities = compute_boundary_node_features(
    boundary_transactions,
    community_nodes
)

nodes_communities

wash_trading_846
features shape: (2820, 20)


community,address,label,internal_out_tx,internal_out_value_eth,unique_internal_receivers,internal_in_tx,internal_in_value_eth,unique_internal_senders,incoming_from_external_tx,incoming_from_external_value_eth,unique_external_senders,outgoing_to_external_tx,outgoing_to_external_value_eth,unique_external_receivers,total_tx_boundary,total_value_eth_boundary,unique_receivers_total,unique_senders_total,unique_counterparties_sum
str,str,str,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,u32
"""wash_trading_846""","""0x003ac1f90ee1a46b07186ce9ceac…","""normal""",1,0.215012,1,1,0.215017,1,0,0.0,0,0,0.0,0,2,0.43003,1,1,2
"""wash_trading_846""","""0x0062d65b5f9b850d18338f697301…","""normal""",8,4.542205,4,6,4.542145,4,0,0.0,0,0,0.0,0,14,9.08435,4,4,8
"""wash_trading_846""","""0x0062dbf8caa4d65655548a2e3564…","""normal""",1,1.0000e-9,1,0,0.0,0,0,0.0,0,0,0.0,0,1,1.0000e-9,1,0,1
"""wash_trading_846""","""0x006bbdd3f45ddca912d570e5aee8…","""fraud""",33,31.921533,17,37,31.050478,20,2,0.871192,2,0,0.0,0,72,63.843203,17,22,39
"""wash_trading_846""","""0x006bc22f03973a6632df73cc6bec…","""normal""",3,3.0000e-9,2,0,0.0,0,0,0.0,0,0,0.0,0,3,3.0000e-9,2,0,2
"""wash_trading_846""","""0x00ae468f1bd31c93535c9644302b…","""normal""",1,1.0000e-9,1,0,0.0,0,0,0.0,0,0,0.0,0,1,1.0000e-9,1,0,1
"""wash_trading_846""","""0x00ae4e53707ae92c3f470929bc45…","""normal""",10,7.193622,6,10,7.193652,6,0,0.0,0,0,0.0,0,20,14.387275,6,6,12
"""wash_trading_846""","""0x00c2104559e2ce0921288ee3c99a…","""normal""",7,3.263839,2,8,2.839864,2,0,0.0,0,4,0.14,2,19,6.243702,4,2,6
"""wash_trading_846""","""0x00d1e64a898b1c03e54e82bd23cc…","""normal""",3,1.012129,2,2,1.393805,1,0,0.0,0,5,0.62,2,10,3.025934,4,1,5


phishing_390
features shape: (17500, 20)


community,address,label,internal_out_tx,internal_out_value_eth,unique_internal_receivers,internal_in_tx,internal_in_value_eth,unique_internal_senders,incoming_from_external_tx,incoming_from_external_value_eth,unique_external_senders,outgoing_to_external_tx,outgoing_to_external_value_eth,unique_external_receivers,total_tx_boundary,total_value_eth_boundary,unique_receivers_total,unique_senders_total,unique_counterparties_sum
str,str,str,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,u32
"""phishing_390""","""0x000bf4254dbaf6acb4809ec1511a…","""normal""",1,0.000155,1,1,0.000168,1,0,0.0,0,1,0.0,1,3,0.000323,2,1,3
"""phishing_390""","""0x000d55e4bbb6b8cdff19ef2dec1a…","""normal""",1,0.000145,1,1,0.000151,1,0,0.0,0,1,0.0,1,3,0.000296,2,1,3
"""phishing_390""","""0x000da24bda94cc1c59e7b339ab06…","""normal""",2,0.000747,1,3,0.003483,1,0,0.0,0,3,0.0,1,8,0.00423,2,1,3
"""phishing_390""","""0x00136a2e22f8ed17b9a15e800572…","""normal""",2,0.000707,1,2,0.000725,1,0,0.0,0,2,0.0,1,6,0.001432,2,1,3
"""phishing_390""","""0x0014539b3bcd161e60f2086b04af…","""normal""",1,0.000325,1,1,0.000334,1,0,0.0,0,1,0.0,1,3,0.000659,2,1,3
"""phishing_390""","""0x0015050f65858bc4de26d111c1b9…","""normal""",1,0.000261,1,1,0.000269,1,0,0.0,0,1,0.0,1,3,0.00053,2,1,3
"""phishing_390""","""0x0017621870362696c484f205960f…","""normal""",0,0.0,0,1,0.015773,1,0,0.0,0,0,0.0,0,1,0.015773,0,1,1
"""phishing_390""","""0x00195f521852d0cf848a39645a73…","""normal""",1,0.005968,1,0,0.0,0,1,0.00597,1,0,0.0,0,2,0.011937,1,1,2
"""phishing_390""","""0x0023c0efd0d4fd342236bfea2a7b…","""normal""",1,0.006549,1,0,0.0,0,0,0.0,0,1,0.0,1,2,0.006549,2,0,2


mixer_242
features shape: (5901, 20)


community,address,label,internal_out_tx,internal_out_value_eth,unique_internal_receivers,internal_in_tx,internal_in_value_eth,unique_internal_senders,incoming_from_external_tx,incoming_from_external_value_eth,unique_external_senders,outgoing_to_external_tx,outgoing_to_external_value_eth,unique_external_receivers,total_tx_boundary,total_value_eth_boundary,unique_receivers_total,unique_senders_total,unique_counterparties_sum
str,str,str,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,u32
"""mixer_242""","""0x00024b0cfb6c7ae68408c32e1513…","""normal""",1,0.602284,1,0,0.0,0,1,0.29083,1,0,0.0,0,2,0.893114,1,1,2
"""mixer_242""","""0x001a4c4904ada7acbef177b80859…","""normal""",0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0
"""mixer_242""","""0x0044fb6b14bc24d45dc82f10913e…","""normal""",0,0.0,0,1,0.002198,1,0,0.0,0,1,0.0,1,2,0.002198,1,1,2
"""mixer_242""","""0x00474678499082d85c07d4b5b07c…","""normal""",0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0
"""mixer_242""","""0x0058cdc8ba2dfa609d266110d202…","""normal""",0,0.0,0,1,0.0028,1,0,0.0,0,4,0.000075,4,5,0.002875,4,1,5
"""mixer_242""","""0x006df42722a3573e26a584e587d1…","""normal""",2,457.070842,1,0,0.0,0,2,457.008129,1,0,0.0,0,4,914.078971,1,1,2
"""mixer_242""","""0x007155d9b9dac297b281413f430c…","""normal""",0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0
"""mixer_242""","""0x009fbd14f4129c12b0cc96400967…","""normal""",0,0.0,0,1,0.0018,1,0,0.0,0,0,0.0,0,1,0.0018,0,1,1
"""mixer_242""","""0x00a4aef2b82dc4b6ea84431a55b0…","""normal""",0,0.0,0,1,0.024631,1,0,0.0,0,0,0.0,0,1,0.024631,0,1,1


community,address,label,internal_out_tx,internal_out_value_eth,unique_internal_receivers,internal_in_tx,internal_in_value_eth,unique_internal_senders,incoming_from_external_tx,incoming_from_external_value_eth,unique_external_senders,outgoing_to_external_tx,outgoing_to_external_value_eth,unique_external_receivers,total_tx_boundary,total_value_eth_boundary,unique_receivers_total,unique_senders_total,unique_counterparties_sum
str,str,str,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,u32
"""mixer_242""","""0x28c6c06298d514db089934071355…","""fraud""",13362,67953.085619,2086,1735,11559.800424,1469,3240,231381.69608,1590,17093,209811.987347,2819,35430,520706.569469,4905,3059,7964
"""mixer_242""","""0xdfd5293d8e347dfe59e90efd55b2…","""normal""",186,8837.393668,133,3,49748.337012,1,1,0.0,1,17716,53203.425957,4644,17906,111789.156638,4777,2,4779
"""mixer_242""","""0xee7ae85f2fe2239e27d9c1e23fff…","""normal""",0,0.0,0,11167,0.0,1,3,0.000475,2,0,0.0,0,11170,0.000475,0,3,3
"""mixer_242""","""0x699ee12a1d97437a4a1e87c71e5d…","""normal""",0,0.0,0,1,13.65545,1,2,304.881094,1,2785,317.537144,3,2788,636.073688,3,2,5
"""mixer_242""","""0x07ae8551be970cb1cca11dd7a11f…","""normal""",4,0.0,2,0,0.0,0,2,561.365328,2,2297,561.282584,13,2303,1122.647912,15,2,17
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""wash_trading_846""","""0x8cbdc87d6d98b9eab82271efade8…","""normal""",0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0
"""wash_trading_846""","""0xe47762a37f31a0b46d694f6e2716…","""normal""",0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0
"""wash_trading_846""","""0xd910f99686f6b26dec76c34a910b…","""normal""",0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0


In [35]:
nodes_communities.write_csv("nodes_communities_base_features.csv")

## 7. Temporal Motif Counting Functions

In [36]:
delta = 3600

communities = [
    "wash_trading_846",
    "phishing_390",
    "mixer_242"
]

### 7.1 Repeated Same Direction Motif

In [37]:
def count_repeated_pairs_in_group(group, delta=3600):
    # Enforce chronological ordering within the specific source-target dyad group
    group = group.sort("timestamp")

    # Extract columns to native Python primitive lists for fast pointer iteration
    timestamps = group["timestamp"].to_list()
    values = group["value_eth"].to_list()

    # Initialize two-pointer sliding window track metrics and accumulation states
    left = 0
    pair_count = 0
    motif_value_sum = 0.0
    window_value_sum = 0.0

    # Expand the right pointer to ingest the chronological transaction stream
    for right in range(len(timestamps)):
        current_time = timestamps[right]
        current_value = values[right]

        # Shrink window from the left to evict records exceeding the temporal delta
        while current_time - timestamps[left] > delta:
            window_value_sum -= values[left] # Subtract evicted asset weight from current window buffer
            left += 1

        # Calculate the number of valid lookback events currently in scope
        previous_count = right - left

        # Accumulate combinatorial counts and historical value pairings
        pair_count += previous_count
        motif_value_sum += window_value_sum + previous_count * current_value

        # Advance current transaction size into the rolling lookback buffer
        window_value_sum += current_value

    # Pack aggregated link characteristics into a single operational Polars row dataframe
    return pl.DataFrame({
        "source": [group["source"][0]],
        "target": [group["target"][0]],
        "repeat_same_direction_pair_count_1h": [pair_count],
        "repeat_same_direction_pair_value_eth_1h": [motif_value_sum]
    })

In [38]:
def compute_repeated_same_direction(
    community,
    tx,
    nodes_communities,
    delta=3600
):
    print("=" * 80)
    print(f"Processing: {community}")

    # Standardize field types, select core tracking features, and optimize sort layout
    tx = (
        tx
        .with_columns([
            pl.col("source").cast(pl.Utf8),
            pl.col("target").cast(pl.Utf8),
            pl.col("timestamp").cast(pl.Int64),
            pl.col("value_eth").cast(pl.Float64)
        ])
        .select(["source", "target", "timestamp", "value_eth"])
        .sort(["source", "target", "timestamp"])
    )

    # Isolate baseline node mappings belonging to the current community evaluation frame
    base = nodes_communities.filter(pl.col("community") == community)

    print(f"Boundary transactions: {tx.height:,}")
    print(f"Base community nodes: {base.height:,}")

    # Map groups using the sliding-window function over each individual sender-receiver dyad
    dyad_features = (
        tx
        .group_by("source", "target")
        .map_groups(lambda group: count_repeated_pairs_in_group(group, delta=delta))
        .filter(pl.col("repeat_same_direction_pair_count_1h") > 0)
    )

    print(f"Dyads with repeated same-direction motifs: {dyad_features.height:,}")

    if dyad_features.height > 0:
        print(
            "Total repeated same-direction pairs:",
            dyad_features["repeat_same_direction_pair_count_1h"].sum()
        )
    else:
        print("Total repeated same-direction pairs: 0")

    # Aggregate counts and economic values for nodes acting as directional loop origin senders
    sender_features = (
        dyad_features
        .group_by("source")
        .agg([
            pl.col("repeat_same_direction_pair_count_1h")
            .sum()
            .alias("repeat_same_direction_as_sender_count_1h"),

            pl.col("repeat_same_direction_pair_value_eth_1h")
            .sum()
            .alias("repeat_same_direction_as_sender_value_eth_1h")
        ])
        .rename({"source": "address"})
    )

    # Aggregate counts and economic values for nodes acting as directional loop target receivers
    receiver_features = (
        dyad_features
        .group_by("target")
        .agg([
            pl.col("repeat_same_direction_pair_count_1h")
            .sum()
            .alias("repeat_same_direction_as_receiver_count_1h"),

            pl.col("repeat_same_direction_pair_value_eth_1h")
            .sum()
            .alias("repeat_same_direction_as_receiver_value_eth_1h")
        ])
        .rename({"target": "address"})
    )

    # Compile all structural features back to the community layout, imputing zero for missing states
    node_features = (
        base
        .select(["community", "address", "label", "total_tx_boundary"])
        .join(sender_features, on="address", how="left")
        .join(receiver_features, on="address", how="left")
        .with_columns([
            pl.col("repeat_same_direction_as_sender_count_1h").fill_null(0),
            pl.col("repeat_same_direction_as_sender_value_eth_1h").fill_null(0),
            pl.col("repeat_same_direction_as_receiver_count_1h").fill_null(0),
            pl.col("repeat_same_direction_as_receiver_value_eth_1h").fill_null(0)
        ])
        # Compute combined interaction frequencies and asset volume weights
        .with_columns([
            (
                pl.col("repeat_same_direction_as_sender_count_1h") +
                pl.col("repeat_same_direction_as_receiver_count_1h")
            ).alias("repeat_same_direction_total_count_1h"),

            (
                pl.col("repeat_same_direction_as_sender_value_eth_1h") +
                pl.col("repeat_same_direction_as_receiver_value_eth_1h")
            ).alias("repeat_same_direction_total_value_eth_1h")
        ])
        # Calculate localized behavior densities normalized against total activity limits
        .with_columns([
            pl.when(pl.col("total_tx_boundary") > 0)
            .then(pl.col("repeat_same_direction_total_count_1h") / pl.col("total_tx_boundary"))
            .otherwise(0)
            .alias("repeat_same_direction_per_tx_1h")
        ])
    )

    # Generate high-level tracking diagnostics comparing normal vs. fraud behavioral classes
    summary = (
        node_features
        .group_by("label")
        .agg([
            pl.len().alias("n_nodes"),

            (pl.col("repeat_same_direction_total_count_1h") > 0)
            .sum()
            .alias("active_repeat_nodes"),

            (
                (pl.col("repeat_same_direction_total_count_1h") > 0).sum() / pl.len()
            ).alias("active_repeat_share"),

            pl.col("repeat_same_direction_total_count_1h").mean().alias("mean_repeat_total_count_1h"),
            pl.col("repeat_same_direction_total_count_1h").median().alias("median_repeat_total_count_1h"),
            pl.col("repeat_same_direction_total_count_1h").max().alias("max_repeat_total_count_1h"),

            pl.col("repeat_same_direction_as_sender_count_1h").mean().alias("mean_repeat_as_sender_1h"),
            pl.col("repeat_same_direction_as_receiver_count_1h").mean().alias("mean_repeat_as_receiver_1h"),

            pl.col("repeat_same_direction_per_tx_1h").mean().alias("mean_repeat_per_tx_1h"),
            pl.col("repeat_same_direction_per_tx_1h").median().alias("median_repeat_per_tx_1h"),

            pl.col("repeat_same_direction_total_value_eth_1h").mean().alias("mean_repeat_value_eth_1h"),
            pl.col("repeat_same_direction_total_value_eth_1h").median().alias("median_repeat_value_eth_1h")
        ])
        .with_columns(pl.lit(community).alias("community"))
        .select([
            "community",
            "label",
            "n_nodes",
            "active_repeat_nodes",
            "active_repeat_share",
            "mean_repeat_total_count_1h",
            "median_repeat_total_count_1h",
            "max_repeat_total_count_1h",
            "mean_repeat_as_sender_1h",
            "mean_repeat_as_receiver_1h",
            "mean_repeat_per_tx_1h",
            "median_repeat_per_tx_1h",
            "mean_repeat_value_eth_1h",
            "median_repeat_value_eth_1h"
        ])
        .sort("label")
    )

    return node_features, summary

### 7.2 Reciprocity Motif

In [39]:
def compute_reciprocity_for_community(
    community,
    tx,
    nodes_communities,
    delta=3600
):
    print("=" * 80)
    print(f"Processing reciprocity: {community}")

    # Standardize types, retain core tracking dimensions, and pre-sort by directional coordinates
    tx = (
        tx
        .with_columns([
            pl.col("hash").cast(pl.Utf8),
            pl.col("source").cast(pl.Utf8),
            pl.col("target").cast(pl.Utf8),
            pl.col("timestamp").cast(pl.Int64),
            pl.col("value_eth").cast(pl.Float64)
        ])
        .select(["hash", "source", "target", "timestamp", "value_eth"])
        .sort(["source", "target", "timestamp"])
    )

    # Isolate reference node metadata belonging explicitly to the current target community
    base = nodes_communities.filter(pl.col("community") == community)

    print(f"Boundary transactions: {tx.height:,}")
    print(f"Base community nodes: {base.height:,}")

    # Create an alias instance of the ledger representing the initial triggering transactions (A → B)
    tx_first = tx.select([
        pl.col("hash").alias("hash_1"),
        pl.col("source").alias("source_1"),
        pl.col("target").alias("target_1"),
        pl.col("timestamp").alias("timestamp_1"),
        pl.col("value_eth").alias("value_eth_1")
    ])

    # Create an alias instance of the ledger representing the returning response transactions (B → A)
    tx_second = tx.select([
        pl.col("hash").alias("hash_2"),
        pl.col("source").alias("source_2"),
        pl.col("target").alias("target_2"),
        pl.col("timestamp").alias("timestamp_2"),
        pl.col("value_eth").alias("value_eth_2")
    ])

    # Intersect self-joined streams where the sender-receiver coordinates are explicitly flipped (A → B and B → A)
    reciprocity_pairs = (
        tx_first
        .join(
            tx_second,
            left_on=["source_1", "target_1"],
            right_on=["target_2", "source_2"],
            how="inner"
        )
        # Apply filters to enforce chronological ordering and the strict temporal window delta constraint
        .filter(
            (pl.col("timestamp_2") > pl.col("timestamp_1")) &
            ((pl.col("timestamp_2") - pl.col("timestamp_1")) <= delta)
        )
        # Extract the precise duration latency gap and calculate combined transaction volumes
        .with_columns([
            (pl.col("timestamp_2") - pl.col("timestamp_1")).alias("delta_seconds"),
            (pl.col("value_eth_1") + pl.col("value_eth_2")).alias("motif_value_eth")
        ])
    )

    print(f"Reciprocity pairs: {reciprocity_pairs.height:,}")

    # Aggregate occurrence counts and value distributions for nodes acting as the initial trigger sender
    first_sender_features = (
        reciprocity_pairs
        .group_by("source_1")
        .agg([
            pl.len().alias("reciprocity_as_first_sender_count_1h"),
            pl.col("motif_value_eth").sum().alias("reciprocity_as_first_sender_value_eth_1h")
        ])
        .rename({"source_1": "address"})
    )

    # Aggregate occurrence counts and value distributions for nodes acting as the initial trigger receiver
    first_receiver_features = (
        reciprocity_pairs
        .group_by("target_1")
        .agg([
            pl.len().alias("reciprocity_as_first_receiver_count_1h"),
            pl.col("motif_value_eth").sum().alias("reciprocity_as_first_receiver_value_eth_1h")
        ])
        .rename({"target_1": "address"})
    )

    # Map the generated dyadic features back into the central node layout, safely fill missing entries with 0
    node_features = (
        base
        .select(["community", "address", "label", "total_tx_boundary"])
        .join(first_sender_features, on="address", how="left")
        .join(first_receiver_features, on="address", how="left")
        .with_columns([
            pl.col("reciprocity_as_first_sender_count_1h").fill_null(0),
            pl.col("reciprocity_as_first_sender_value_eth_1h").fill_null(0),
            pl.col("reciprocity_as_first_receiver_count_1h").fill_null(0),
            pl.col("reciprocity_as_first_receiver_value_eth_1h").fill_null(0)
        ])
        # Compute multi-role combined statistics across individual account addresses
        .with_columns([
            (
                pl.col("reciprocity_as_first_sender_count_1h") +
                pl.col("reciprocity_as_first_receiver_count_1h")
            ).alias("reciprocity_total_count_1h"),

            (
                pl.col("reciprocity_as_first_sender_value_eth_1h") +
                pl.col("reciprocity_as_first_receiver_value_eth_1h")
            ).alias("reciprocity_total_value_eth_1h")
        ])
        # Derive structural footprint metrics normalized by the aggregate boundary transaction volumes
        .with_columns([
            pl.when(pl.col("total_tx_boundary") > 0)
            .then(pl.col("reciprocity_total_count_1h") / pl.col("total_tx_boundary"))
            .otherwise(0)
            .alias("reciprocity_per_tx_1h")
        ])
    )

    # Compute descriptive statistical diagnostics to contrast normal vs. fraud populations
    summary = (
        node_features
        .group_by("label")
        .agg([
            pl.len().alias("n_nodes"),

            (pl.col("reciprocity_total_count_1h") > 0)
            .sum()
            .alias("active_reciprocity_nodes"),

            (
                (pl.col("reciprocity_total_count_1h") > 0).sum() / pl.len()
            ).alias("active_reciprocity_share"),

            pl.col("reciprocity_total_count_1h").mean().alias("mean_reciprocity_total_count_1h"),
            pl.col("reciprocity_total_count_1h").median().alias("median_reciprocity_total_count_1h"),
            pl.col("reciprocity_total_count_1h").max().alias("max_reciprocity_total_count_1h"),

            pl.col("reciprocity_as_first_sender_count_1h").mean().alias("mean_reciprocity_as_first_sender_1h"),
            pl.col("reciprocity_as_first_receiver_count_1h").mean().alias("mean_reciprocity_as_first_receiver_1h"),

            pl.col("reciprocity_per_tx_1h").mean().alias("mean_reciprocity_per_tx_1h"),
            pl.col("reciprocity_per_tx_1h").median().alias("median_reciprocity_per_tx_1h"),

            pl.col("reciprocity_total_value_eth_1h").mean().alias("mean_reciprocity_value_eth_1h"),
            pl.col("reciprocity_total_value_eth_1h").median().alias("median_reciprocity_value_eth_1h")
        ])
        .with_columns(pl.lit(community).alias("community"))
        .select([
            # Structural coordinates mapping index
            "community",
            "label",
            "n_nodes",
            # Behavior footprint coverage dimensions
            "active_reciprocity_nodes",
            "active_reciprocity_share",
            # Total frequency distribution descriptors
            "mean_reciprocity_total_count_1h",
            "median_reciprocity_total_count_1h",
            "max_reciprocity_total_count_1h",
            # Directional trigger-role profiles
            "mean_reciprocity_as_first_sender_1h",
            "mean_reciprocity_as_first_receiver_1h",
            # Density metrics normalized against transaction frequency
            "mean_reciprocity_per_tx_1h",
            "median_reciprocity_per_tx_1h",
            # Financial economic weight distributions
            "mean_reciprocity_value_eth_1h",
            "median_reciprocity_value_eth_1h"
        ])
        .sort("label")
    )

    return node_features, summary

### 7.3 Chain Motif

In [40]:
def empty_chain_contribs():
    # Return an empty schema-aligned DataFrame fallback for groups with no motif interactions
    return pl.DataFrame({
        "address": pl.Series([], dtype=pl.Utf8),
        "chain_as_start_count_1h": pl.Series([], dtype=pl.Int64),
        "chain_as_start_value_eth_1h": pl.Series([], dtype=pl.Float64),
        "chain_as_middle_count_1h": pl.Series([], dtype=pl.Int64),
        "chain_as_middle_value_eth_1h": pl.Series([], dtype=pl.Float64),
        "chain_as_end_count_1h": pl.Series([], dtype=pl.Int64),
        "chain_as_end_value_eth_1h": pl.Series([], dtype=pl.Float64)
    })


def count_chain_contribs_for_middle(group, delta=3600):
    # Establish the focal pivot node (B) from the current grouped interaction state
    middle = group["middle"][0]

    # Split and chronologically sort incoming edges (A → B)
    incoming = (
        group
        .filter(pl.col("direction") == "incoming")
        .sort("timestamp")
    )

    # Split and chronologically sort outgoing edges (B → C)
    outgoing = (
        group
        .filter(pl.col("direction") == "outgoing")
        .sort("timestamp")
    )

    # Fast-path exit if a valid chain pattern is structurally impossible
    if incoming.height == 0 or outgoing.height == 0:
        return empty_chain_contribs()

    # Cast incoming arrays to native lists for optimized index pointer lookups
    in_sources = incoming["counterparty"].to_list()
    in_times = incoming["timestamp"].to_list()
    in_values = incoming["value_eth"].to_list()

    # Cast outgoing arrays to native lists for optimized index pointer lookups
    out_targets = outgoing["counterparty"].to_list()
    out_times = outgoing["timestamp"].to_list()
    out_values = outgoing["value_eth"].to_list()

    # Map structure: address -> [start_count, start_val, middle_count, middle_val, end_count, end_val]
    contrib = defaultdict(lambda: [0, 0.0, 0, 0.0, 0, 0.0])

    # Scan each incoming event to detect subsequent casual outgoing paths
    for i in range(len(in_times)):
        a = in_sources[i]
        t_in = in_times[i]
        v_in = in_values[i]

        # Use binary search to isolate the lookback slice within the timestamp constraint
        left = bisect_right(out_times, t_in)
        right = bisect_right(out_times, t_in + delta)

        # Match with valid outgoing targets (C) inside the active window
        for j in range(left, right):
            c = out_targets[j]

            # Reject cycles and self-loops to preserve strict 3-node graph chain logic
            if a == middle or middle == c or a == c:
                continue

            v_out = out_values[j]
            motif_value = v_in + v_out

            # Log execution metrics tracking node role positioning weights
            contrib[a][0] += 1
            contrib[a][1] += motif_value

            contrib[middle][2] += 1
            contrib[middle][3] += motif_value

            contrib[c][4] += 1
            contrib[c][5] += motif_value

    if len(contrib) == 0:
        return empty_chain_contribs()

    # Convert hashmap storage records to list structure format
    rows = []
    for address, values in contrib.items():
        rows.append((
            address,
            values[0],
            values[1],
            values[2],
            values[3],
            values[4],
            values[5]
        ))

    # Re-wrap structured metrics inside an explicitly typed Polars dataframe
    return pl.DataFrame(
        rows,
        schema=[
            "address",
            "chain_as_start_count_1h",
            "chain_as_start_value_eth_1h",
            "chain_as_middle_count_1h",
            "chain_as_middle_value_eth_1h",
            "chain_as_end_count_1h",
            "chain_as_end_value_eth_1h"
        ],
        orient="row"
    )

In [41]:
def compute_chain_for_community(
    community,
    tx,
    nodes_communities,
    delta=3600
):
    print("=" * 80)
    print(f"Processing chain: {community}")

    # Standardize data types and clean self-loops from raw boundary entries
    tx = (
        tx
        .with_columns([
            pl.col("source").cast(pl.Utf8),
            pl.col("target").cast(pl.Utf8),
            pl.col("timestamp").cast(pl.Int64),
            pl.col("value_eth").cast(pl.Float64)
        ])
        .select(["source", "target", "timestamp", "value_eth"])
        .filter(pl.col("source") != pl.col("target"))
    )

    # Isolate relevant subset nodes from global tracking mappings
    base = nodes_communities.filter(pl.col("community") == community)

    print(f"Boundary transactions: {tx.height:,}")
    print(f"Base community nodes: {base.height:,}")

    # Format incoming transaction records relative to target receiver nodes
    incoming_events = (
        tx
        .select([
            pl.col("target").alias("middle"),
            pl.col("source").alias("counterparty"),
            pl.col("timestamp"),
            pl.col("value_eth")
        ])
        .with_columns(pl.lit("incoming").alias("direction"))
    )

    # Format outgoing transaction records relative to source originator nodes
    outgoing_events = (
        tx
        .select([
            pl.col("source").alias("middle"),
            pl.col("target").alias("counterparty"),
            pl.col("timestamp"),
            pl.col("value_eth")
        ])
        .with_columns(pl.lit("outgoing").alias("direction"))
    )

    # Merge events and execute custom group matching routine around the pivot nodes
    events = pl.concat([incoming_events, outgoing_events])

    chain_contribs_raw = (
        events
        .group_by("middle")
        .map_groups(lambda group: count_chain_contribs_for_middle(group, delta=delta))
    )

    # Consolidate role attributes across multi-group network outputs
    if chain_contribs_raw.height == 0:
        chain_features = empty_chain_contribs()
    else:
        chain_features = (
            chain_contribs_raw
            .group_by("address")
            .agg([
                pl.col("chain_as_start_count_1h").sum(),
                pl.col("chain_as_start_value_eth_1h").sum(),
                pl.col("chain_as_middle_count_1h").sum(),
                pl.col("chain_as_middle_value_eth_1h").sum(),
                pl.col("chain_as_end_count_1h").sum(),
                pl.col("chain_as_end_value_eth_1h").sum()
            ])
        )

    print(f"Nodes participating in chain motifs: {chain_features.height:,}")

    # Build master features schema layout, computing role counts and volume densities
    node_features = (
        base
        .select(["community", "address", "label", "total_tx_boundary"])
        .join(chain_features, on="address", how="left")
        .with_columns([
            pl.col("chain_as_start_count_1h").fill_null(0),
            pl.col("chain_as_start_value_eth_1h").fill_null(0),
            pl.col("chain_as_middle_count_1h").fill_null(0),
            pl.col("chain_as_middle_value_eth_1h").fill_null(0),
            pl.col("chain_as_end_count_1h").fill_null(0),
            pl.col("chain_as_end_value_eth_1h").fill_null(0)
        ])
        .with_columns([
            (
                pl.col("chain_as_start_count_1h") +
                pl.col("chain_as_middle_count_1h") +
                pl.col("chain_as_end_count_1h")
            ).alias("chain_total_count_1h"),

            (
                pl.col("chain_as_start_value_eth_1h") +
                pl.col("chain_as_middle_value_eth_1h") +
                pl.col("chain_as_end_value_eth_1h")
            ).alias("chain_total_value_eth_1h")
        ])
        .with_columns([
            pl.when(pl.col("total_tx_boundary") > 0)
            .then(pl.col("chain_total_count_1h") / pl.col("total_tx_boundary"))
            .otherwise(0)
            .alias("chain_per_tx_1h")
        ])
    )

    # Extract behavioral category statistics comparing normal vs. fraudulent entities
    summary = (
        node_features
        .group_by("label")
        .agg([
            pl.len().alias("n_nodes"),

            (pl.col("chain_total_count_1h") > 0)
            .sum()
            .alias("active_chain_nodes"),

            ((pl.col("chain_total_count_1h") > 0).sum() / pl.len())
            .alias("active_chain_share"),

            pl.col("chain_total_count_1h").mean().alias("mean_chain_total_count_1h"),
            pl.col("chain_total_count_1h").median().alias("median_chain_total_count_1h"),
            pl.col("chain_total_count_1h").max().alias("max_chain_total_count_1h"),

            pl.col("chain_as_start_count_1h").mean().alias("mean_chain_as_start_1h"),
            pl.col("chain_as_middle_count_1h").mean().alias("mean_chain_as_middle_1h"),
            pl.col("chain_as_end_count_1h").mean().alias("mean_chain_as_end_1h"),

            pl.col("chain_per_tx_1h").mean().alias("mean_chain_per_tx_1h"),
            pl.col("chain_per_tx_1h").median().alias("median_chain_per_tx_1h"),

            pl.col("chain_total_value_eth_1h").mean().alias("mean_chain_value_eth_1h"),
            pl.col("chain_total_value_eth_1h").median().alias("median_chain_value_eth_1h")
        ])
        .with_columns(pl.lit(community).alias("community"))
        .select([
            "community",
            "label",
            "n_nodes",
            "active_chain_nodes",
            "active_chain_share",
            "mean_chain_total_count_1h",
            "median_chain_total_count_1h",
            "max_chain_total_count_1h",
            "mean_chain_as_start_1h",
            "mean_chain_as_middle_1h",
            "mean_chain_as_end_1h",
            "mean_chain_per_tx_1h",
            "median_chain_per_tx_1h",
            "mean_chain_value_eth_1h",
            "median_chain_value_eth_1h"
        ])
        .sort("label")
    )

    print(f"Rows: {node_features.height:,}")
    print(f"Columns: {len(node_features.columns):,}")

    return node_features, summary

### 7.4 Fan-In Motif

In [42]:
def empty_fan_in_contribs():
    # Return an empty schema-aligned DataFrame fallback for groups with no fan-in motif components
    return pl.DataFrame({
        "address": pl.Series([], dtype=pl.Utf8),
        "fan_in_as_sender_count_1h": pl.Series([], dtype=pl.Int64),
        "fan_in_as_sender_value_eth_1h": pl.Series([], dtype=pl.Float64),
        "fan_in_as_center_count_1h": pl.Series([], dtype=pl.Int64),
        "fan_in_as_center_value_eth_1h": pl.Series([], dtype=pl.Float64)
    })


def count_fan_in_contribs_for_center(group, delta=3600):
    # Order transactions sequentially by timeline timestamps within the focal receiver group
    group = group.sort("timestamp")

    # Isolate the target consolidation endpoint center (C) and parse features to native arrays
    center = group["target"][0]
    sources = group["source"].to_list()
    times = group["timestamp"].to_list()
    values = group["value_eth"].to_list()

    # Map tracking structure: address -> [sender_count, sender_value, center_count, center_value]
    contrib = defaultdict(lambda: [0, 0.0, 0, 0.0])

    # Initialize two-pointer parameters alongside separate state trace trackers for each unique sender source
    left = 0
    window_count = 0
    window_value_sum = 0.0
    source_counts = defaultdict(int)
    source_value_sums = defaultdict(float)

    # Expand the sliding window pointer rightward across the timeline array
    for right in range(len(times)):
        current_time = times[right]
        current_source = sources[right]
        current_value = values[right]

        # Shift the left window boundary pointer forward to drop outdated historical events
        while current_time - times[left] > delta:
            old_source = sources[left]
            old_value = values[left]

            # Evict metrics parameters from both global sliding structures and source maps
            window_count -= 1
            window_value_sum -= old_value
            source_counts[old_source] -= 1
            source_value_sums[old_source] -= old_value

            # Clean memory registries for inactive sources with no active links left
            if source_counts[old_source] == 0:
                del source_counts[old_source]
                del source_value_sums[old_source]

            left += 1

        # Extract transaction trace frequencies and volumes belonging specifically to the current source
        same_source_count = source_counts.get(current_source, 0)
        same_source_value_sum = source_value_sums.get(current_source, 0.0)

        # Isolate valid non-identical peer interactions to satisfy distinct 3-node fan-in topology
        valid_previous_count = window_count - same_source_count
        valid_previous_value_sum = window_value_sum - same_source_value_sum

        # Aggregate parameters if a valid fan-in structure exists in the current lookback slice
        if valid_previous_count > 0:
            current_sender_value = valid_previous_value_sum + valid_previous_count * current_value

            # Update structural profiles for the incoming transaction originator address role
            contrib[current_source][0] += valid_previous_count
            contrib[current_source][1] += current_sender_value

            # Update structural profiles for the central focal target destination address role
            contrib[center][2] += valid_previous_count
            contrib[center][3] += current_sender_value

            # Back-propagate motif values onto all separate valid peer sources within the sliding layout
            for previous_source, previous_count in source_counts.items():
                if previous_source == current_source:
                    continue

                previous_value_sum = source_value_sums[previous_source]
                previous_sender_value = previous_value_sum + previous_count * current_value

                contrib[previous_source][0] += previous_count
                contrib[previous_source][1] += previous_sender_value

        # Insert current data parameters into running sliding window status buffers
        window_count += 1
        window_value_sum += current_value
        source_counts[current_source] += 1
        source_value_sums[current_source] += current_value

    if len(contrib) == 0:
        return empty_fan_in_contribs()

    # Reformat metrics dictionary arrays into a list of tracking tuples
    rows = []
    for address, values_list in contrib.items():
        rows.append((
            address,
            values_list[0],
            values_list[1],
            values_list[2],
            values_list[3]
        ))

    # Commit processed profiles back to a structured schema-aligned Polars DataFrame
    return pl.DataFrame(
        rows,
        schema=[
            "address",
            "fan_in_as_sender_count_1h",
            "fan_in_as_sender_value_eth_1h",
            "fan_in_as_center_count_1h",
            "fan_in_as_center_value_eth_1h"
        ],
        orient="row"
    )

In [43]:
def compute_fan_in_for_community(
    community,
    tx,
    nodes_communities,
    delta=3600
):
    print("=" * 80)
    print(f"Processing fan-in: {community}")

    # Standardize explicit fields, filter self-loops, and perform sorting optimized for aggregation targets
    tx = (
        tx
        .with_columns([
            pl.col("source").cast(pl.Utf8),
            pl.col("target").cast(pl.Utf8),
            pl.col("timestamp").cast(pl.Int64),
            pl.col("value_eth").cast(pl.Float64)
        ])
        .select(["source", "target", "timestamp", "value_eth"])
        .filter(pl.col("source") != pl.col("target"))
        .sort(["target", "timestamp"])
    )

    # Isolate relevant baseline structural identities mapping to the current target cluster
    base = nodes_communities.filter(pl.col("community") == community)

    print(f"Boundary transactions: {tx.height:,}")
    print(f"Base community nodes: {base.height:,}")

    # Map groups using the sliding-window function over each targeted destination sink node (center)
    fan_in_contribs_raw = (
        tx
        .group_by("target")
        .map_groups(lambda group: count_fan_in_contribs_for_center(group, delta=delta))
    )

    # Condense multi-role outputs into a standardized unique address feature list
    if fan_in_contribs_raw.height == 0:
        fan_in_features = empty_fan_in_contribs()
    else:
        fan_in_features = (
            fan_in_contribs_raw
            .group_by("address")
            .agg([
                pl.col("fan_in_as_sender_count_1h").sum(),
                pl.col("fan_in_as_sender_value_eth_1h").sum(),
                pl.col("fan_in_as_center_count_1h").sum(),
                pl.col("fan_in_as_center_value_eth_1h").sum()
            ])
        )

    print(f"Nodes participating in fan-in motifs: {fan_in_features.height:,}")

    # Align generated temporal arrays with baseline node profiles and fill unmapped elements with zero
    node_features = (
        base
        .select(["community", "address", "label", "total_tx_boundary"])
        .join(fan_in_features, on="address", how="left")
        .with_columns([
            pl.col("fan_in_as_sender_count_1h").fill_null(0),
            pl.col("fan_in_as_sender_value_eth_1h").fill_null(0),
            pl.col("fan_in_as_center_count_1h").fill_null(0),
            pl.col("fan_in_as_center_value_eth_1h").fill_null(0)
        ])
        # Compute combined frequency and asset metrics across distinct network role designations
        .with_columns([
            (
                pl.col("fan_in_as_sender_count_1h") +
                pl.col("fan_in_as_center_count_1h")
            ).alias("fan_in_total_count_1h"),

            (
                pl.col("fan_in_as_sender_value_eth_1h") +
                pl.col("fan_in_as_center_value_eth_1h")
            ).alias("fan_in_total_value_eth_1h")
        ])
        # Calculate dynamic footprint metrics normalized against entire localized boundary dimensions
        .with_columns([
            pl.when(pl.col("total_tx_boundary") > 0)
            .then(pl.col("fan_in_total_count_1h") / pl.col("total_tx_boundary"))
            .otherwise(0)
            .alias("fan_in_per_tx_1h")
        ])
    )

    # Compute high-level tracking diagnostics comparing normal vs. fraud behavioral classes
    summary = (
        node_features
        .group_by("label")
        .agg([
            pl.len().alias("n_nodes"),

            (pl.col("fan_in_total_count_1h") > 0)
            .sum()
            .alias("active_fan_in_nodes"),

            ((pl.col("fan_in_total_count_1h") > 0).sum() / pl.len())
            .alias("active_fan_in_share"),

            pl.col("fan_in_total_count_1h").mean().alias("mean_fan_in_total_count_1h"),
            pl.col("fan_in_total_count_1h").median().alias("median_fan_in_total_count_1h"),
            pl.col("fan_in_total_count_1h").max().alias("max_fan_in_total_count_1h"),

            pl.col("fan_in_as_sender_count_1h").mean().alias("mean_fan_in_as_sender_1h"),
            pl.col("fan_in_as_center_count_1h").mean().alias("mean_fan_in_as_center_1h"),

            pl.col("fan_in_per_tx_1h").mean().alias("mean_fan_in_per_tx_1h"),
            pl.col("fan_in_per_tx_1h").median().alias("median_fan_in_per_tx_1h"),

            pl.col("fan_in_total_value_eth_1h").mean().alias("mean_fan_in_value_eth_1h"),
            pl.col("fan_in_total_value_eth_1h").median().alias("median_fan_in_value_eth_1h")
        ])
        .with_columns(pl.lit(community).alias("community"))
        .select([
            # Structural coordinate mappings index
            "community",
            "label",
            "n_nodes",
            # Behavior pattern footprint coverage dimensions
            "active_fan_in_nodes",
            "active_fan_in_share",
            # Total frequency distribution descriptors
            "mean_fan_in_total_count_1h",
            "median_fan_in_total_count_1h",
            "max_fan_in_total_count_1h",
            # Role position profiles inside fan-in patterns
            "mean_fan_in_as_sender_1h",
            "mean_fan_in_as_center_1h",
            # Density variables normalized against execution rates
            "mean_fan_in_per_tx_1h",
            "median_fan_in_per_tx_1h",
            # Financial asset allocation concentrations
            "mean_fan_in_value_eth_1h",
            "median_fan_in_value_eth_1h"
        ])
        .sort("label")
    )

    print(f"Rows: {node_features.height:,}")
    print(f"Columns: {len(node_features.columns):,}")

    return node_features, summary

### 7.5 Fan-Out Motif

In [44]:
def empty_fan_out_contribs():
    # Return an empty schema-aligned DataFrame fallback for groups with no fan-out motif components
    return pl.DataFrame({
        "address": pl.Series([], dtype=pl.Utf8),
        "fan_out_as_center_count_1h": pl.Series([], dtype=pl.Int64),
        "fan_out_as_center_value_eth_1h": pl.Series([], dtype=pl.Float64),
        "fan_out_as_receiver_count_1h": pl.Series([], dtype=pl.Int64),
        "fan_out_as_receiver_value_eth_1h": pl.Series([], dtype=pl.Float64)
    })


def count_fan_out_contribs_for_center(group, delta=3600):
    # Order transactions sequentially by timeline timestamps within the focal sender group
    group = group.sort("timestamp")

    # Isolate the source originator endpoint center (A) and parse features to native arrays
    center = group["source"][0]
    targets = group["target"].to_list()
    times = group["timestamp"].to_list()
    values = group["value_eth"].to_list()

    # Map tracking structure: address -> [center_count, center_value, receiver_count, receiver_value]
    contrib = defaultdict(lambda: [0, 0.0, 0, 0.0])

    # Initialize two-pointer parameters alongside separate state trace trackers for each unique target receiver
    left = 0
    window_count = 0
    window_value_sum = 0.0
    target_counts = defaultdict(int)
    target_value_sums = defaultdict(float)

    # Expand the sliding window pointer rightward across the timeline array
    for right in range(len(times)):
        current_time = times[right]
        current_target = targets[right]
        current_value = values[right]

        # Shift the left window boundary pointer forward to drop outdated historical events
        while current_time - times[left] > delta:
            old_target = targets[left]
            old_value = values[left]

            # Evict metrics parameters from both global sliding structures and target maps
            window_count -= 1
            window_value_sum -= old_value
            target_counts[old_target] -= 1
            target_value_sums[old_target] -= old_value

            # Clean memory registries for inactive targets with no active links left
            if target_counts[old_target] == 0:
                del target_counts[old_target]
                del target_value_sums[old_target]

            left += 1

        # Extract transaction trace frequencies and volumes belonging specifically to the current target
        same_target_count = target_counts.get(current_target, 0)
        same_target_value_sum = target_value_sums.get(current_target, 0.0)

        # Isolate valid non-identical peer interactions to satisfy distinct 3-node fan-out topology
        valid_previous_count = window_count - same_target_count
        valid_previous_value_sum = window_value_sum - same_target_value_sum

        # Aggregate parameters if a valid fan-out structure exists in the current lookback slice
        if valid_previous_count > 0:
            current_receiver_value = valid_previous_value_sum + valid_previous_count * current_value

            # Update structural profiles for the central focal distributor center source address role
            contrib[center][0] += valid_previous_count
            contrib[center][1] += current_receiver_value

            # Update structural profiles for the current leaf receiver target address role
            contrib[current_target][2] += valid_previous_count
            contrib[current_target][3] += current_receiver_value

            # Back-propagate motif values onto all separate valid peer targets within the sliding layout
            for previous_target, previous_count in target_counts.items():
                if previous_target == current_target:
                    continue

                previous_value_sum = target_value_sums[previous_target]
                previous_receiver_value = previous_value_sum + previous_count * current_value

                contrib[previous_target][2] += previous_count
                contrib[previous_target][3] += previous_receiver_value

        # Insert current data parameters into running sliding window status buffers
        window_count += 1
        window_value_sum += current_value
        target_counts[current_target] += 1
        target_value_sums[current_target] += current_value

    if len(contrib) == 0:
        return empty_fan_out_contribs()

    # Reformat metrics dictionary arrays into a list of tracking tuples
    rows = []
    for address, values_list in contrib.items():
        rows.append((
            address,
            values_list[0],
            values_list[1],
            values_list[2],
            values_list[3]
        ))

    # Commit processed profiles back to a structured schema-aligned Polars DataFrame
    return pl.DataFrame(
        rows,
        schema=[
            "address",
            "fan_out_as_center_count_1h",
            "fan_out_as_center_value_eth_1h",
            "fan_out_as_receiver_count_1h",
            "fan_out_as_receiver_value_eth_1h"
        ],
        orient="row"
    )

In [45]:
def compute_fan_out_for_community(
    community,
    tx,
    nodes_communities,
    delta=3600
):
    print("=" * 80)
    print(f"Processing fan-out: {community}")

    # Standardize explicit fields, filter self-loops, and perform sorting optimized for aggregation centers
    tx = (
        tx
        .with_columns([
            pl.col("source").cast(pl.Utf8),
            pl.col("target").cast(pl.Utf8),
            pl.col("timestamp").cast(pl.Int64),
            pl.col("value_eth").cast(pl.Float64)
        ])
        .select(["source", "target", "timestamp", "value_eth"])
        .filter(pl.col("source") != pl.col("target"))
        .sort(["source", "timestamp"])
    )

    # Isolate relevant baseline structural identities mapping to the current target cluster
    base = nodes_communities.filter(pl.col("community") == community)

    print(f"Boundary transactions: {tx.height:,}")
    print(f"Base community nodes: {base.height:,}")

    # Map groups using the sliding-window function over each targeted origin source node (center)
    fan_out_contribs_raw = (
        tx
        .group_by("source")
        .map_groups(lambda group: count_fan_out_contribs_for_center(group, delta=delta))
    )

    # Condense multi-role outputs into a standardized unique address feature list
    if fan_out_contribs_raw.height == 0:
        fan_out_features = empty_fan_out_contribs()
    else:
        fan_out_features = (
            fan_out_contribs_raw
            .group_by("address")
            .agg([
                pl.col("fan_out_as_center_count_1h").sum(),
                pl.col("fan_out_as_center_value_eth_1h").sum(),
                pl.col("fan_out_as_receiver_count_1h").sum(),
                pl.col("fan_out_as_receiver_value_eth_1h").sum()
            ])
        )

    print(f"Nodes participating in fan-out motifs: {fan_out_features.height:,}")

    # Align generated temporal arrays with baseline node profiles and fill unmapped elements with zero
    node_features = (
        base
        .select(["community", "address", "label", "total_tx_boundary"])
        .join(fan_out_features, on="address", how="left")
        .with_columns([
            pl.col("fan_out_as_center_count_1h").fill_null(0),
            pl.col("fan_out_as_center_value_eth_1h").fill_null(0),
            pl.col("fan_out_as_receiver_count_1h").fill_null(0),
            pl.col("fan_out_as_receiver_value_eth_1h").fill_null(0)
        ])
        # Compute combined frequency and asset metrics across distinct network role designations
        .with_columns([
            (
                pl.col("fan_out_as_center_count_1h") +
                pl.col("fan_out_as_receiver_count_1h")
            ).alias("fan_out_total_count_1h"),

            (
                pl.col("fan_out_as_center_value_eth_1h") +
                pl.col("fan_out_as_receiver_value_eth_1h")
            ).alias("fan_out_total_value_eth_1h")
        ])
        # Calculate dynamic footprint metrics normalized against entire localized boundary dimensions
        .with_columns([
            pl.when(pl.col("total_tx_boundary") > 0)
            .then(pl.col("fan_out_total_count_1h") / pl.col("total_tx_boundary"))
            .otherwise(0)
            .alias("fan_out_per_tx_1h")
        ])
    )

    # Compute high-level tracking diagnostics comparing normal vs. fraud behavioral classes
    summary = (
        node_features
        .group_by("label")
        .agg([
            pl.len().alias("n_nodes"),

            (pl.col("fan_out_total_count_1h") > 0)
            .sum()
            .alias("active_fan_out_nodes"),

            ((pl.col("fan_out_total_count_1h") > 0).sum() / pl.len())
            .alias("active_fan_out_share"),

            pl.col("fan_out_total_count_1h").mean().alias("mean_fan_out_total_count_1h"),
            pl.col("fan_out_total_count_1h").median().alias("median_fan_out_total_count_1h"),
            pl.col("fan_out_total_count_1h").max().alias("max_fan_out_total_count_1h"),

            pl.col("fan_out_as_center_count_1h").mean().alias("mean_fan_out_as_center_1h"),
            pl.col("fan_out_as_receiver_count_1h").mean().alias("mean_fan_out_as_receiver_1h"),

            pl.col("fan_out_per_tx_1h").mean().alias("mean_fan_out_per_tx_1h"),
            pl.col("fan_out_per_tx_1h").median().alias("median_fan_out_per_tx_1h"),

            pl.col("fan_out_total_value_eth_1h").mean().alias("mean_fan_out_value_eth_1h"),
            pl.col("fan_out_total_value_eth_1h").median().alias("median_fan_out_value_eth_1h")
        ])
        .with_columns(pl.lit(community).alias("community"))
        .select([
            # Structural coordinate mappings index
            "community",
            "label",
            "n_nodes",
            # Behavior pattern footprint coverage dimensions
            "active_fan_out_nodes",
            "active_fan_out_share",
            # Total frequency distribution descriptors
            "mean_fan_out_total_count_1h",
            "median_fan_out_total_count_1h",
            "max_fan_out_total_count_1h",
            # Role position profiles inside fan-out patterns
            "mean_fan_out_as_center_1h",
            "mean_fan_out_as_receiver_1h",
            # Density variables normalized against execution rates
            "mean_fan_out_per_tx_1h",
            "median_fan_out_per_tx_1h",
            # Financial asset allocation concentrations
            "mean_fan_out_value_eth_1h",
            "median_fan_out_value_eth_1h"
        ])
        .sort("label")
    )

    print(f"Rows: {node_features.height:,}")
    print(f"Columns: {len(node_features.columns):,}")

    return node_features, summary

### 7.6 Cycle Motif

In [46]:
def empty_cycle_contribs():
    # Return an empty schema-aligned DataFrame fallback for groups with no cycle motif components
    return pl.DataFrame({
        "address": pl.Series([], dtype=pl.Utf8),
        "cycle_as_start_count_1h": pl.Series([], dtype=pl.Int64),
        "cycle_as_start_value_eth_1h": pl.Series([], dtype=pl.Float64),
        "cycle_as_middle_count_1h": pl.Series([], dtype=pl.Int64),
        "cycle_as_middle_value_eth_1h": pl.Series([], dtype=pl.Float64),
        "cycle_as_end_count_1h": pl.Series([], dtype=pl.Int64),
        "cycle_as_end_value_eth_1h": pl.Series([], dtype=pl.Float64)
    })

def build_cycle_indexes(tx):
    # Initialize index mappings for fast lookup of incoming transactions and cycle-closing pairs
    incoming_by_target = defaultdict(lambda: {"times": [], "sources": [], "values": []})
    closing_by_pair = defaultdict(lambda: {"times": [], "values": []})

    # Sort the transaction stream chronologically to ensure binary search and prefix sum compatibility
    tx_sorted = tx.sort("timestamp")

    # Populate the lookup tables with timestamps, node addresses, and transaction volumes
    for source, target, timestamp, value_eth in tx_sorted.iter_rows():
        incoming_by_target[target]["times"].append(timestamp)
        incoming_by_target[target]["sources"].append(source)
        incoming_by_target[target]["values"].append(value_eth)

        closing_by_pair[(source, target)]["times"].append(timestamp)
        closing_by_pair[(source, target)]["values"].append(value_eth)

    closing_prefix_by_pair = {}

    # Construct prefix sums over transaction values for each node pair to achieve O(1) interval value summation
    for pair, data in closing_by_pair.items():
        prefix = [0.0]
        for value in data["values"]:
            prefix.append(prefix[-1] + value)
        closing_prefix_by_pair[pair] = prefix

    return incoming_by_target, closing_by_pair, closing_prefix_by_pair

In [47]:
def compute_cycle_contribs(tx, delta=3600):
    # Build lookback index mappings and prefix value arrays for fast lookup
    incoming_by_target, closing_by_pair, closing_prefix_by_pair = build_cycle_indexes(tx)

    # Map tracking structure: address -> [start_count, start_val, middle_count, middle_val, end_count, end_val]
    contrib = defaultdict(lambda: [0, 0.0, 0, 0.0, 0, 0.0])

    tx_second_edges = tx.sort("timestamp")
    n_second_edges = tx_second_edges.height

    # Iterate over every transaction treating it as the intermediate second edge (B -> C) of the cycle
    for idx, row in enumerate(tx_second_edges.iter_rows(named=True)):
        b = row["source"]
        c = row["target"]
        t2 = row["timestamp"]
        v2 = row["value_eth"]

        incoming_data = incoming_by_target.get(b)
        if incoming_data is None:
            continue

        in_times = incoming_data["times"]
        in_sources = incoming_data["sources"]
        in_values = incoming_data["values"]

        # Identify preceding incoming edges (A -> B) that fall within the valid lookback window delta
        left_in = bisect_left(in_times, t2 - delta)
        right_in = bisect_left(in_times, t2)

        if left_in == right_in:
            continue

        # Evaluate every valid causal parent transaction candidate
        for i in range(left_in, right_in):
            a = in_sources[i]
            t1 = in_times[i]
            v1 = in_values[i]

            # Exclude self-loops and dual-node cycles to maintain strict 3-node cycle topology criteria
            if a == b or b == c or a == c:
                continue

            closing_pair = (c, a)
            closing_data = closing_by_pair.get(closing_pair)
            if closing_data is None:
                continue

            close_times = closing_data["times"]
            close_prefix = closing_prefix_by_pair[closing_pair]

            # Locate subsequent matching edges (C -> A) that close the temporal loop within the active window constraint
            left_close = bisect_right(close_times, t2)
            right_close = bisect_right(close_times, t1 + delta)

            k = right_close - left_close
            if k <= 0:
                continue

            # Calculate the total asset value transferred throughout this specific motif instance loop
            closing_value_sum = close_prefix[right_close] - close_prefix[left_close]
            motif_value_sum = k * (v1 + v2) + closing_value_sum

            # Accumulate count weights and total monetary values across all three participating node roles
            contrib[a][0] += k
            contrib[a][1] += motif_value_sum

            contrib[b][2] += k
            contrib[b][3] += motif_value_sum

            contrib[c][4] += k
            contrib[c][5] += motif_value_sum

    if len(contrib) == 0:
        return empty_cycle_contribs()

    # Reformat metrics dictionary arrays into a list of tracking tuples
    rows = []
    for address, values in contrib.items():
        rows.append((
            address,
            values[0],
            values[1],
            values[2],
            values[3],
            values[4],
            values[5]
        ))

    # Commit processed profiles back to a structured schema-aligned Polars DataFrame
    return pl.DataFrame(
        rows,
        schema=[
            "address",
            "cycle_as_start_count_1h",
            "cycle_as_start_value_eth_1h",
            "cycle_as_middle_count_1h",
            "cycle_as_middle_value_eth_1h",
            "cycle_as_end_count_1h",
            "cycle_as_end_value_eth_1h"
        ],
        orient="row"
    )

In [48]:
def compute_cycle_for_community(
    community,
    tx,
    nodes_communities,
    delta=3600
):
    print("=" * 80)
    print(f"Processing cycle: {community}")

    # Standardize data types and clean self-loops from raw boundary entries
    tx = (
        tx
        .with_columns([
            pl.col("source").cast(pl.Utf8),
            pl.col("target").cast(pl.Utf8),
            pl.col("timestamp").cast(pl.Int64),
            pl.col("value_eth").cast(pl.Float64)
        ])
        .select(["source", "target", "timestamp", "value_eth"])
        .filter(pl.col("source") != pl.col("target"))
    )

    # Isolate relevant baseline structural identities mapping to the current target cluster
    base = nodes_communities.filter(pl.col("community") == community)

    print(f"Boundary transactions: {tx.height:,}")
    print(f"Base community nodes: {base.height:,}")

    # Process localized data using the specialized cycle extraction function
    cycle_features = compute_cycle_contribs(tx, delta=delta)

    print(f"Nodes participating in cycle motifs: {cycle_features.height:,}")

    # Align generated temporal arrays with baseline node profiles and fill unmapped elements with zero
    node_features = (
        base
        .select(["community", "address", "label", "total_tx_boundary"])
        .join(cycle_features, on="address", how="left")
        .with_columns([
            pl.col("cycle_as_start_count_1h").fill_null(0),
            pl.col("cycle_as_start_value_eth_1h").fill_null(0),
            pl.col("cycle_as_middle_count_1h").fill_null(0),
            pl.col("cycle_as_middle_value_eth_1h").fill_null(0),
            pl.col("cycle_as_end_count_1h").fill_null(0),
            pl.col("cycle_as_end_value_eth_1h").fill_null(0)
        ])
        # Compute combined frequency and asset metrics across distinct network role designations
        .with_columns([
            (
                pl.col("cycle_as_start_count_1h") +
                pl.col("cycle_as_middle_count_1h") +
                pl.col("cycle_as_end_count_1h")
            ).alias("cycle_total_count_1h"),

            (
                pl.col("cycle_as_start_value_eth_1h") +
                pl.col("cycle_as_middle_value_eth_1h") +
                pl.col("cycle_as_end_value_eth_1h")
            ).alias("cycle_total_value_eth_1h")
        ])
        # Calculate dynamic footprint metrics normalized against entire localized boundary dimensions
        .with_columns([
            pl.when(pl.col("total_tx_boundary") > 0)
            .then(pl.col("cycle_total_count_1h") / pl.col("total_tx_boundary"))
            .otherwise(0)
            .alias("cycle_per_tx_1h")
        ])
    )

    # Compute descriptive statistical diagnostics to contrast normal vs. fraud populations
    summary = (
        node_features
        .group_by("label")
        .agg([
            pl.len().alias("n_nodes"),

            (pl.col("cycle_total_count_1h") > 0)
            .sum()
            .alias("active_cycle_nodes"),

            ((pl.col("cycle_total_count_1h") > 0).sum() / pl.len())
            .alias("active_cycle_share"),

            pl.col("cycle_total_count_1h").mean().alias("mean_cycle_total_count_1h"),
            pl.col("cycle_total_count_1h").median().alias("median_cycle_total_count_1h"),
            pl.col("cycle_total_count_1h").max().alias("max_cycle_total_count_1h"),

            pl.col("cycle_as_start_count_1h").mean().alias("mean_cycle_as_start_1h"),
            pl.col("cycle_as_middle_count_1h").mean().alias("mean_cycle_as_middle_1h"),
            pl.col("cycle_as_end_count_1h").mean().alias("mean_cycle_as_end_1h"),

            pl.col("cycle_per_tx_1h").mean().alias("mean_cycle_per_tx_1h"),
            pl.col("cycle_per_tx_1h").median().alias("median_cycle_per_tx_1h"),

            pl.col("cycle_total_value_eth_1h").mean().alias("mean_cycle_value_eth_1h"),
            pl.col("cycle_total_value_eth_1h").median().alias("median_cycle_value_eth_1h")
        ])
        .with_columns(pl.lit(community).alias("community"))
        .select([
            # Structural coordinates mapping index
            "community",
            "label",
            "n_nodes",
            # Behavior pattern footprint coverage dimensions
            "active_cycle_nodes",
            "active_cycle_share",
            # Total frequency distribution descriptors
            "mean_cycle_total_count_1h",
            "median_cycle_total_count_1h",
            "max_cycle_total_count_1h",
            # Role position profiles inside cycle patterns
            "mean_cycle_as_start_1h",
            "mean_cycle_as_middle_1h",
            "mean_cycle_as_end_1h",
            # Density metrics normalized against transaction frequency
            "mean_cycle_per_tx_1h",
            "median_cycle_per_tx_1h",
            # Financial economic weight distributions
            "mean_cycle_value_eth_1h",
            "median_cycle_value_eth_1h"
        ])
        .sort("label")
    )

    print(f"Rows: {node_features.height:,}")
    print(f"Columns: {len(node_features.columns):,}")

    return node_features, summary

### 7.7 Computing Motifs

In [49]:
def compute_motif_for_all_communities(
    motif_name,
    compute_function,
    feature_cols,
    communities,
    boundary_transactions,
    nodes_communities,
    delta=3600
):
    # Initialize list collectors to hold localized per-community data partitions
    all_features = []
    all_summaries = []

    print("=" * 100)
    print(f"Computing motif: {motif_name}")

    # Sequentially iterate through the partition index to isolate and process target clusters
    for community in communities:
        # Execute the motif tracking function callback passing the pre-filtered boundary transaction dictionary map
        node_features, summary = compute_function(
            community=community,
            tx=boundary_transactions[community],
            nodes_communities=nodes_communities,
            delta=delta
        )

        # Slice the core keys along with the requested configuration columns
        motif_features = node_features.select(
            ["community", "address"] + feature_cols
        )

        # Append individual outputs to the temporary list buffers
        all_features.append(motif_features)
        all_summaries.append(summary)

    # Vertically stack the array collections into centralized, multi-community master DataFrames
    features_all = pl.concat(all_features)
    summary_all = pl.concat(all_summaries)

    return features_all, summary_all

def add_motif_features_to_nodes(
    nodes_communities,
    motif_features_all,
    feature_cols
):
    # Map the processed metrics back into the master table using a multi-key left join
    nodes_communities = (
        nodes_communities
        .join(
            motif_features_all,
            on=["community", "address"],
            how="left"
        )
        # Handle unmapped structural elements by filling NaN/null entries with zero
        .with_columns([
            pl.col(col).fill_null(0)
            for col in feature_cols
        ])
    )

    return nodes_communities

## 8. Final Feature Table Creation

In [50]:
motif_configs = {
    # 2-Node Pattern: Sequential repeated transfers matching identical orientation (A → B, then A → B)
    "repeated_same_direction": {
        "compute_function": compute_repeated_same_direction,
        "feature_cols": [
            "repeat_same_direction_as_sender_count_1h",
            "repeat_same_direction_as_sender_value_eth_1h",
            "repeat_same_direction_as_receiver_count_1h",
            "repeat_same_direction_as_receiver_value_eth_1h",
            "repeat_same_direction_total_count_1h",
            "repeat_same_direction_total_value_eth_1h",
            "repeat_same_direction_per_tx_1h"
        ]
    },

    # 2-Node Pattern: Chronological bilateral exchange loop coordinates (A → B, then B → A)
    "reciprocity": {
        "compute_function": compute_reciprocity_for_community,
        "feature_cols": [
            "reciprocity_as_first_sender_count_1h",
            "reciprocity_as_first_sender_value_eth_1h",
            "reciprocity_as_first_receiver_count_1h",
            "reciprocity_as_first_receiver_value_eth_1h",
            "reciprocity_total_count_1h",
            "reciprocity_total_value_eth_1h",
            "reciprocity_per_tx_1h"
        ]
    },

    # 3-Node Pattern: Linear transactional sequence propagation channels (A → B → C)
    "chain": {
        "compute_function": compute_chain_for_community,
        "feature_cols": [
            "chain_as_start_count_1h",
            "chain_as_start_value_eth_1h",
            "chain_as_middle_count_1h",
            "chain_as_middle_value_eth_1h",
            "chain_as_end_count_1h",
            "chain_as_end_value_eth_1h",
            "chain_total_count_1h",
            "chain_total_value_eth_1h",
            "chain_per_tx_1h"
        ]
    },

    # 3-Node Pattern: Multi-source funds consolidation into a single repository center (A → C and B → C)
    "fan_in": {
        "compute_function": compute_fan_in_for_community,
        "feature_cols": [
            "fan_in_as_sender_count_1h",
            "fan_in_as_sender_value_eth_1h",
            "fan_in_as_center_count_1h",
            "fan_in_as_center_value_eth_1h",
            "fan_in_total_count_1h",
            "fan_in_total_value_eth_1h",
            "fan_in_per_tx_1h"
        ]
    },

    # 3-Node Pattern: Single-source wallet dispersal branching out to multiple leaves (A → B and A → C)
    "fan_out": {
        "compute_function": compute_fan_out_for_community,
        "feature_cols": [
            "fan_out_as_center_count_1h",
            "fan_out_as_center_value_eth_1h",
            "fan_out_as_receiver_count_1h",
            "fan_out_as_receiver_value_eth_1h",
            "fan_out_total_count_1h",
            "fan_out_total_value_eth_1h",
            "fan_out_per_tx_1h"
        ]
    },

    # 3-Node Pattern: Closed chronological circular liquidity loop system (A → B → C → A)
    "cycle": {
        "compute_function": compute_cycle_for_community,
        "feature_cols": [
            "cycle_as_start_count_1h",
            "cycle_as_start_value_eth_1h",
            "cycle_as_middle_count_1h",
            "cycle_as_middle_value_eth_1h",
            "cycle_as_end_count_1h",
            "cycle_as_end_value_eth_1h",
            "cycle_total_count_1h",
            "cycle_total_value_eth_1h",
            "cycle_per_tx_1h"
        ]
    }
}

In [51]:
# Initialize storage maps to keep standalone copies of raw results per motif type
motif_features_results = {}
motif_summary_results = {}

# Iteratively execute, aggregate, and merge metrics for every temporal motif configuration
for motif_name, config in motif_configs.items():
    # Run the core multi-community processor for the active motif pattern lookups
    features_all, summary_all = compute_motif_for_all_communities(
        motif_name=motif_name,
        compute_function=config["compute_function"],
        feature_cols=config["feature_cols"],
        communities=communities,
        boundary_transactions=boundary_transactions,
        nodes_communities=nodes_communities,
        delta=delta
    )

    # Persist intermediate structures into independent tracking dictionaries
    motif_features_results[motif_name] = features_all
    motif_summary_results[motif_name] = summary_all

    # Sequentially append the newly generated feature metrics onto the primary nodes table
    nodes_communities = add_motif_features_to_nodes(
        nodes_communities=nodes_communities,
        motif_features_all=features_all,
        feature_cols=config["feature_cols"]
    )

Computing motif: repeated_same_direction
Processing: wash_trading_846
Boundary transactions: 33,681
Base community nodes: 2,820
Dyads with repeated same-direction motifs: 2,331
Total repeated same-direction pairs: 28132
Processing: phishing_390
Boundary transactions: 65,504
Base community nodes: 17,500
Dyads with repeated same-direction motifs: 1,460
Total repeated same-direction pairs: 619931
Processing: mixer_242
Boundary transactions: 70,569
Base community nodes: 5,901
Dyads with repeated same-direction motifs: 1,643
Total repeated same-direction pairs: 3185282
Computing motif: reciprocity
Processing reciprocity: wash_trading_846
Boundary transactions: 33,681
Base community nodes: 2,820
Reciprocity pairs: 25,207
Processing reciprocity: phishing_390
Boundary transactions: 65,504
Base community nodes: 17,500
Reciprocity pairs: 212
Processing reciprocity: mixer_242
Boundary transactions: 70,569
Base community nodes: 5,901
Reciprocity pairs: 73
Computing motif: chain
Processing chain: w

In [52]:
nodes_communities.write_csv("nodes_communities_with_all_motifs.csv")

print("Saved: nodes_communities_with_all_motifs.csv")
print("Rows:", nodes_communities.height)
print("Columns:", len(nodes_communities.columns))

Saved: nodes_communities_with_all_motifs.csv
Rows: 26221
Columns: 66
